In [ ]:
%pip install -q monai SimpleITK nibabel scikit-image trimesh

In [ ]:
import torch
import monai
import SimpleITK as sitk
import nibabel
import skimage
import trimesh

print("PyTorch:", torch.__version__)
print("MONAI:", monai.__version__)
print("SimpleITK:", sitk.Version_VersionString())
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("All required libraries imported successfully!")

In [ ]:
import os

paths = [
    "/kaggle/working/PENGWIN/downloads/ct_part1.zip",
    "/kaggle/working/PENGWIN/downloads/ct_part2.zip",
    "/kaggle/working/PENGWIN/downloads/labels.zip",
    "/kaggle/working/PENGWIN/checkpoints/fragment_aware_epoch_25.pth"
]

for path in paths:
    if os.path.exists(path):
        size_gb = os.path.getsize(path) / (1024**3)
        print(f"FOUND: {path} ({size_gb:.2f} GB)")
    else:
        print(f"NOT FOUND: {path}")

In [ ]:
import os
import requests

BASE = "/kaggle/working/PENGWIN"
DOWNLOAD_DIR = f"{BASE}/downloads"
CHECKPOINT_DIR = f"{BASE}/checkpoints"

os.makedirs(DOWNLOAD_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

files = {
    "ct_part1.zip":
        "https://zenodo.org/records/10927452/files/PENGWIN_CT_train_images_part1.zip?download=1",
    "ct_part2.zip":
        "https://zenodo.org/records/10927452/files/PENGWIN_CT_train_images_part2.zip?download=1",
    "labels.zip":
        "https://zenodo.org/records/10927452/files/PENGWIN_CT_train_labels.zip?download=1"
}

for filename, url in files.items():
    path = os.path.join(DOWNLOAD_DIR, filename)

    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"Already exists: {filename}")
        continue

    print(f"Downloading {filename}...")

    with requests.get(url, stream=True, timeout=(30, 300)) as response:
        response.raise_for_status()

        with open(path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)

    print(f"Downloaded: {filename}")

print("\nDownload step finished.")

In [ ]:
import os
import zipfile

DOWNLOAD_DIR = "/kaggle/working/PENGWIN/downloads"

files = [
    "ct_part1.zip",
    "ct_part2.zip",
    "labels.zip"
]

for filename in files:
    path = os.path.join(DOWNLOAD_DIR, filename)

    if not os.path.exists(path):
        print(f"MISSING: {filename}")
        continue

    size_gb = os.path.getsize(path) / (1024**3)
    print(f"\nFile: {filename}")
    print(f"Size: {size_gb:.2f} GB")

    try:
        with zipfile.ZipFile(path, "r") as z:
            bad_file = z.testzip()

            if bad_file is None:
                print("ZIP integrity: OK")
                print("Number of files:", len(z.namelist()))
            else:
                print("ZIP integrity: FAILED at", bad_file)

    except zipfile.BadZipFile:
        print("ZIP integrity: INVALID")

In [ ]:
import zipfile
import os

DOWNLOAD_DIR = "/kaggle/working/PENGWIN/downloads"

for filename in ["ct_part1.zip", "ct_part2.zip", "labels.zip"]:
    path = os.path.join(DOWNLOAD_DIR, filename)

    print(f"\n--- {filename} ---")

    with zipfile.ZipFile(path, "r") as z:
        for name in z.namelist()[:5]:
            print(name)

        print("Total files:", len(z.namelist()))

In [ ]:
import os
import zipfile
import SimpleITK as sitk
import numpy as np

BASE = "/kaggle/working/PENGWIN"
DOWNLOAD_DIR = f"{BASE}/downloads"
TEMP_DIR = f"{BASE}/sample_check"

os.makedirs(TEMP_DIR, exist_ok=True)

# Read case 001 directly from the ZIP files
with zipfile.ZipFile(f"{DOWNLOAD_DIR}/ct_part1.zip") as z:
    with z.open("001.mha") as src, open(f"{TEMP_DIR}/001_ct.mha", "wb") as dst:
        dst.write(src.read())

with zipfile.ZipFile(f"{DOWNLOAD_DIR}/labels.zip") as z:
    with z.open("001.mha") as src, open(f"{TEMP_DIR}/001_label.mha", "wb") as dst:
        dst.write(src.read())

# Load CT and ground-truth label
ct_image = sitk.ReadImage(f"{TEMP_DIR}/001_ct.mha")
label_image = sitk.ReadImage(f"{TEMP_DIR}/001_label.mha")

ct = sitk.GetArrayFromImage(ct_image)
label = sitk.GetArrayFromImage(label_image)

print("CT shape:", ct.shape)
print("Label shape:", label.shape)
print("CT spacing:", ct_image.GetSpacing())
print("Label spacing:", label_image.GetSpacing())
print("Ground-truth classes:", np.unique(label))
print("Shapes match:", ct.shape == label.shape)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Find a slice containing labeled fracture fragments
slice_counts = np.count_nonzero(label, axis=(1, 2))
z = int(np.argmax(slice_counts))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].imshow(ct[z], cmap="gray")
axes[0].set_title(f"CT Scan — Slice {z}")
axes[0].axis("off")

axes[1].imshow(label[z], cmap="nipy_spectral", interpolation="nearest")
axes[1].set_title("Ground-Truth Fragments")
axes[1].axis("off")

axes[2].imshow(ct[z], cmap="gray")
axes[2].imshow(
    np.ma.masked_where(label[z] == 0, label[z]),
    cmap="nipy_spectral",
    alpha=0.6,
    interpolation="nearest"
)
axes[2].set_title("CT + Fragment Labels")
axes[2].axis("off")

plt.tight_layout()
plt.show()

print("Selected slice:", z)
print("Fragment labels in this slice:", np.unique(label[z]))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Normalize CT intensity to the range [0, 1]
ct_clipped = np.clip(ct.astype(np.float32), -200, 2000)
ct_normalized = (ct_clipped + 200) / 2200.0

# Keep the original integer fragment labels
label_processed = label.astype(np.int16)

print("CT shape:", ct_normalized.shape)
print("Normalized CT range:",
      float(ct_normalized.min()),
      float(ct_normalized.max()))

print("Label shape:", label_processed.shape)
print("Labels:", np.unique(label_processed))

# Count voxels belonging to each class
classes, counts = np.unique(label_processed, return_counts=True)

print("\nLabel distribution:")
for class_id, count in zip(classes, counts):
    print(f"Class {class_id}: {count:,} voxels")

In [ ]:
import random
import json
import os

SEED = 42
random.seed(SEED)

case_ids = [f"{i:03d}" for i in range(1, 101)]
random.shuffle(case_ids)

train_ids = case_ids[:70]
val_ids = case_ids[70:85]
test_ids = case_ids[85:100]

split = {
    "seed": SEED,
    "train": train_ids,
    "validation": val_ids,
    "test": test_ids
}

split_path = "/kaggle/working/PENGWIN/dataset_split.json"

with open(split_path, "w") as f:
    json.dump(split, f, indent=2)

print("Training cases:", len(train_ids))
print("Validation cases:", len(val_ids))
print("Testing cases:", len(test_ids))
print("First 5 training cases:", train_ids[:5])
print("First 5 validation cases:", val_ids[:5])
print("First 5 testing cases:", test_ids[:5])
print("Split saved to:", split_path)

In [ ]:
import torch
from monai.networks.nets import UNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=31,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1,
    norm="INSTANCE"
).to(device)

print("Device:", device)
print("Model: 3D U-Net")
print("Input channels: 1")
print("Output classes: 31")
print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# Create a dummy 3D patch for a quick model check
dummy_input = torch.rand(1, 1, 96, 96, 96).to(device)

model.eval()

with torch.no_grad():
    dummy_output = model(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", dummy_output.shape)
print("Model test passed:", dummy_output.shape == (1, 31, 96, 96, 96))

del dummy_input, dummy_output
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
import os
import zipfile
import numpy as np
import SimpleITK as sitk
import torch

BASE = "/kaggle/working/PENGWIN"
DOWNLOAD_DIR = f"{BASE}/downloads"
TEMP_DIR = f"{BASE}/real_patch_test"

os.makedirs(TEMP_DIR, exist_ok=True)

case_id = "043"

# Extract case 043 CT
ct_zip = f"{DOWNLOAD_DIR}/ct_part1.zip" if int(case_id) <= 50 else f"{DOWNLOAD_DIR}/ct_part2.zip"

with zipfile.ZipFile(ct_zip) as z:
    with z.open(f"{case_id}.mha") as src, open(f"{TEMP_DIR}/ct.mha", "wb") as dst:
        dst.write(src.read())

# Extract matching ground-truth label
with zipfile.ZipFile(f"{DOWNLOAD_DIR}/labels.zip") as z:
    with z.open(f"{case_id}.mha") as src, open(f"{TEMP_DIR}/label.mha", "wb") as dst:
        dst.write(src.read())

ct_image = sitk.ReadImage(f"{TEMP_DIR}/ct.mha")
label_image = sitk.ReadImage(f"{TEMP_DIR}/label.mha")

ct = sitk.GetArrayFromImage(ct_image).astype(np.float32)
label = sitk.GetArrayFromImage(label_image).astype(np.int16)

print("CT shape:", ct.shape)
print("Label shape:", label.shape)
print("Spacing:", ct_image.GetSpacing())
print("Fragment classes:", np.unique(label))

In [ ]:
import numpy as np
import torch

# Normalize CT intensities
ct_normalized = (
    np.clip(ct, -200, 2000) + 200
) / 2200.0

# Find a voxel belonging to a fracture fragment
coords = np.argwhere(label > 0)

if len(coords) == 0:
    raise ValueError("No fracture fragments found.")

# Select a fragment voxel
z, y, x = coords[np.random.randint(len(coords))]

# Extract a 96 x 96 x 96 patch, handling volume boundaries
patch_size = 96
starts = [
    max(0, min(int(z) - patch_size // 2, ct.shape[0] - patch_size)),
    max(0, min(int(y) - patch_size // 2, ct.shape[1] - patch_size)),
    max(0, min(int(x) - patch_size // 2, ct.shape[2] - patch_size))
]

z0, y0, x0 = starts

ct_patch = ct_normalized[
    z0:z0+96, y0:y0+96, x0:x0+96
]
label_patch = label[
    z0:z0+96, y0:y0+96, x0:x0+96
]

# Convert to tensors: batch, channel, depth, height, width
input_tensor = torch.from_numpy(ct_patch.copy()).float()[None, None].to(device)
label_tensor = torch.from_numpy(label_patch.copy()).long()[None].to(device)

model.eval()
with torch.no_grad():
    output = model(input_tensor)

print("Input shape:", input_tensor.shape)
print("Ground-truth shape:", label_tensor.shape)
print("Prediction shape:", output.shape)
print("Labels in patch:", torch.unique(label_tensor).cpu().tolist())
print("Real-patch test passed:", output.shape == (1, 31, 96, 96, 96))

In [ ]:
import torch
from monai.losses import DiceCELoss

# Combined segmentation loss
loss_function = DiceCELoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True,
    lambda_dice=1.0,
    lambda_ce=1.0
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

print("Loss: Dice + Cross-Entropy")
print("Optimizer: AdamW")
print("Learning rate:", 2e-4)
print("Device:", device)
print("Training configuration ready!")

In [ ]:
model.train()

optimizer.zero_grad()

outputs = model(input_tensor)
loss = loss_function(outputs, label_tensor.unsqueeze(1))

print("Output shape:", outputs.shape)
print("Loss value:", loss.item())

loss.backward()
optimizer.step()

print("Backpropagation successful!")
print("One training step completed.")

In [ ]:
import zipfile
import os

ct_zip_paths = [
    os.path.join(DOWNLOAD_DIR, "ct_part1.zip"),
    os.path.join(DOWNLOAD_DIR, "ct_part2.zip")
]

label_zip_path = os.path.join(DOWNLOAD_DIR, "labels.zip")

print("CT ZIP files exist:",
      all(os.path.exists(p) for p in ct_zip_paths))

print("Label ZIP exists:",
      os.path.exists(label_zip_path))

for path in ct_zip_paths + [label_zip_path]:
    print(os.path.basename(path),
          f"{os.path.getsize(path) / (1024**3):.2f} GB"
          if os.path.exists(path) else "MISSING")

In [ ]:
import os
import zipfile
import numpy as np
import SimpleITK as sitk

TEMP_DIR = os.path.join(BASE, "case_loader_test")
os.makedirs(TEMP_DIR, exist_ok=True)

def load_case(case_id):
    ct_name = f"{case_id}.mha"
    label_name = f"{case_id}.mha"

    # Find the CT scan in either CT ZIP
    ct_data = None
    for zip_path in ct_zip_paths:
        with zipfile.ZipFile(zip_path, "r") as z:
            matches = [
                name for name in z.namelist()
                if os.path.basename(name) == ct_name
            ]
            if matches:
                ct_data = z.read(matches[0])
                break

    if ct_data is None:
        raise FileNotFoundError(f"CT scan {ct_name} not found")

    # Find the corresponding label
    with zipfile.ZipFile(label_zip_path, "r") as z:
        matches = [
            name for name in z.namelist()
            if os.path.basename(name) == label_name
        ]
        if not matches:
            raise FileNotFoundError(f"Label {label_name} not found")
        label_data = z.read(matches[0])

    # Save only this case temporarily
    ct_path = os.path.join(TEMP_DIR, "ct.mha")
    label_path = os.path.join(TEMP_DIR, "label.mha")

    with open(ct_path, "wb") as f:
        f.write(ct_data)

    with open(label_path, "wb") as f:
        f.write(label_data)

    # Read CT and label volumes
    ct_image = sitk.ReadImage(ct_path)
    label_image = sitk.ReadImage(label_path)

    ct_array = sitk.GetArrayFromImage(ct_image)
    label_array = sitk.GetArrayFromImage(label_image)

    assert ct_array.shape == label_array.shape, \
        "CT and label dimensions do not match!"

    return ct_array, label_array


# Test the loader on one training case
test_case_id = train_ids[0]
test_ct, test_label = load_case(test_case_id)

print("Case ID:", test_case_id)
print("CT shape:", test_ct.shape)
print("Label shape:", test_label.shape)
print("Label classes:", np.unique(test_label))
print("Loader test successful!")

In [ ]:
import random
import numpy as np

def sample_training_patch(ct_array, label_array, patch_size=96):
    depth, height, width = ct_array.shape

    # Find voxels belonging to bone fragments
    fragment_voxels = np.argwhere(label_array > 0)

    if len(fragment_voxels) == 0:
        raise ValueError("This CT case has no labeled bone fragments.")

    # Select a random fragment voxel
    z, y, x = fragment_voxels[
        random.randrange(len(fragment_voxels))
    ]

    # Choose a patch around the selected voxel
    def get_start(center, dimension):
        start = center - patch_size // 2
        start = max(0, min(start, dimension - patch_size))
        return start

    z0 = get_start(z, depth)
    y0 = get_start(y, height)
    x0 = get_start(x, width)

    # Extract matching CT and label patches
    ct_patch = ct_array[
        z0:z0 + patch_size,
        y0:y0 + patch_size,
        x0:x0 + patch_size
    ]

    label_patch = label_array[
        z0:z0 + patch_size,
        y0:y0 + patch_size,
        x0:x0 + patch_size
    ]

    # Normalize CT intensities
    ct_patch = np.clip(
        ct_patch.astype(np.float32), -200, 2000
    )
    ct_patch = (ct_patch + 200) / 2200.0

    return ct_patch, label_patch.astype(np.int64)


# Test on case 043
ct_patch, label_patch = sample_training_patch(
    test_ct, test_label
)

print("CT patch shape:", ct_patch.shape)
print("Label patch shape:", label_patch.shape)
print("CT intensity range:",
      float(ct_patch.min()), float(ct_patch.max()))
print("Labels in patch:", np.unique(label_patch))
print("Patch sampler test successful!")

In [ ]:
import torch
import numpy as np

# Sample one patch from case 043
ct_patch, label_patch = sample_training_patch(
    test_ct, test_label, patch_size=96
)

# Convert to PyTorch tensors
input_tensor = torch.from_numpy(
    ct_patch.copy()
).unsqueeze(0).unsqueeze(0).float().to(device)

label_tensor = torch.from_numpy(
    label_patch.copy()
).unsqueeze(0).long().to(device)

print("Input tensor:", input_tensor.shape)
print("Label tensor:", label_tensor.shape)
print("Input device:", input_tensor.device)
print("Unique label classes:", torch.unique(label_tensor).tolist())
print("Batch preparation successful!")

In [ ]:
import torch
import numpy as np
import random
import gc

# Small training test
NUM_CASES = 10
PATCHES_PER_CASE = 1

model.train()
total_loss = 0.0
successful_steps = 0

for case_id in train_ids[:NUM_CASES]:
    try:
        # Load one CT case and its labels
        ct_array, label_array = load_case(case_id)

        # Sample a patch containing bone fragments
        ct_patch, label_patch = sample_training_patch(
            ct_array, label_array, patch_size=96
        )

        # Convert to tensors
        input_tensor = torch.from_numpy(
            ct_patch.copy()
        ).unsqueeze(0).unsqueeze(0).float().to(device)

        label_tensor = torch.from_numpy(
            label_patch.copy()
        ).unsqueeze(0).long().to(device)

        # One optimization step
        optimizer.zero_grad(set_to_none=True)
        outputs = model(input_tensor)
        loss = loss_function(outputs, label_tensor.unsqueeze(1))

        if not torch.isfinite(loss):
            raise ValueError(f"Non-finite loss for case {case_id}")

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        successful_steps += 1

        print(
            f"Case {case_id}: loss = {loss.item():.4f}"
        )

        del ct_array, label_array, ct_patch, label_patch
        del input_tensor, label_tensor, outputs, loss
        gc.collect()
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"Error on case {case_id}: {e}")
        raise

print("\nTraining test completed.")
print("Successful steps:", successful_steps)
print("Average loss:", total_loss / successful_steps)

In [ ]:
import torch
from monai.networks.nets import UNet
from monai.losses import DiceCELoss

# Set reproducible random seeds
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

# Re-create the model with fresh weights
model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=31,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1,
    norm="INSTANCE"
).to(device)

# Re-create the loss function
loss_function = DiceCELoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True,
    lambda_dice=1.0,
    lambda_ce=1.0
)

# Re-create the optimizer
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

print("Baseline model reset successfully!")
print("Trainable parameters:", sum(
    p.numel() for p in model.parameters() if p.requires_grad
))
print("Device:", device)
print("Training cases:", len(train_ids))
print("Validation cases:", len(val_ids))
print("Test cases:", len(test_ids))

In [ ]:
import os
import gc
import torch
import numpy as np

EPOCHS = 1
PATCHES_PER_CASE = 2

CHECKPOINT_DIR = os.path.join(BASE, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model.train()
total_loss = 0.0
steps = 0

for case_index, case_id in enumerate(train_ids, start=1):
    print(f"Loading case {case_index}/{len(train_ids)}: {case_id}")

    ct_array, label_array = load_case(case_id)

    for patch_index in range(PATCHES_PER_CASE):
        ct_patch, label_patch = sample_training_patch(
            ct_array, label_array, patch_size=96
        )

        x = torch.from_numpy(
            ct_patch.copy()
        ).unsqueeze(0).unsqueeze(0).float().to(device)

        y = torch.from_numpy(
            label_patch.copy()
        ).unsqueeze(0).long().to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(x)
        loss = loss_function(outputs, y.unsqueeze(1))

        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss at case {case_id}"
            )

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        steps += 1

        del x, y, outputs, loss

    del ct_array, label_array
    gc.collect()
    torch.cuda.empty_cache()

    if case_index % 10 == 0:
        print(
            f"Processed {case_index} cases | "
            f"Average loss: {total_loss / steps:.4f}"
        )

# Save the trained model and optimizer state
checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_epoch1_pilot.pth"
)

torch.save({
    "epoch": 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "average_training_loss": total_loss / steps,
    "train_ids": train_ids,
    "val_ids": val_ids,
    "test_ids": test_ids
}, checkpoint_path)

print("\nPilot training completed!")
print("Training steps:", steps)
print("Average training loss:", round(total_loss / steps, 4))
print("Checkpoint saved:", checkpoint_path)

In [ ]:
import torch
import numpy as np

# Evaluate one validation case
model.eval()

case_id = val_ids[0]
val_ct, val_label = load_case(case_id)

ct_patch, label_patch = sample_training_patch(
    val_ct, val_label, patch_size=96
)

x = torch.from_numpy(
    ct_patch.copy()
).unsqueeze(0).unsqueeze(0).float().to(device)

with torch.no_grad():
    logits = model(x)
    prediction = torch.argmax(logits, dim=1).squeeze(0).cpu().numpy()

ground_truth = label_patch

# Foreground voxel Dice score
pred_fg = prediction > 0
gt_fg = ground_truth > 0

intersection = np.logical_and(pred_fg, gt_fg).sum()
denominator = pred_fg.sum() + gt_fg.sum()

dice = (
    2.0 * intersection / denominator
    if denominator > 0 else 1.0
)

print("Validation case:", case_id)
print("Prediction shape:", prediction.shape)
print("Ground-truth shape:", ground_truth.shape)
print("Predicted classes:", np.unique(prediction))
print("Foreground Dice:", round(float(dice), 4))

del val_ct, val_label, ct_patch, label_patch, x, logits
torch.cuda.empty_cache()

In [ ]:
import torch
import numpy as np
import gc

model.eval()
dice_scores = []

for case_id in val_ids:
    val_ct, val_label = load_case(case_id)

    ct_patch, label_patch = sample_training_patch(
        val_ct, val_label, patch_size=96
    )

    x = torch.from_numpy(
        ct_patch.copy()
    ).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        prediction = model(x).argmax(dim=1).squeeze(0)
        prediction = prediction.cpu().numpy()

    gt = label_patch
    pred_fg = prediction > 0
    gt_fg = gt > 0

    denominator = pred_fg.sum() + gt_fg.sum()

    if denominator == 0:
        dice = 1.0
    else:
        intersection = np.logical_and(
            pred_fg, gt_fg
        ).sum()
        dice = 2.0 * intersection / denominator

    dice_scores.append(float(dice))
    print(f"Case {case_id}: Dice = {dice:.4f}")

    del val_ct, val_label, ct_patch, label_patch, x
    del prediction, pred_fg, gt_fg
    gc.collect()
    torch.cuda.empty_cache()

print("\nMean validation foreground Dice:",
      round(float(np.mean(dice_scores)), 4))
print("Validation cases evaluated:", len(dice_scores))

In [ ]:
import os
import json
import numpy as np

results = {
    "experiment": "baseline_epoch1_pilot",
    "training_cases": len(train_ids),
    "validation_cases": len(val_ids),
    "test_cases": len(test_ids),
    "patch_size": 96,
    "patches_per_training_case": 2,
    "validation_patches_per_case": 1,
    "metric": "foreground Dice",
    "mean_validation_dice": float(np.mean(dice_scores)),
    "per_case_dice": {
        case_id: float(score)
        for case_id, score in zip(val_ids, dice_scores)
    }
}

results_path = os.path.join(
    BASE, "baseline_validation_pilot.json"
)

with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print("Validation results saved:", results_path)
print("Mean Dice:", results["mean_validation_dice"])

In [ ]:
import SimpleITK as sitk
import numpy as np

case_id = train_ids[0]

ct_path = os.path.join(TEMP_DIR, "ct.mha")
label_path = os.path.join(TEMP_DIR, "label.mha")

# Load the case using your existing loader
ct_array, label_array = load_case(case_id)

# Read metadata from the temporary files
ct_image = sitk.ReadImage(ct_path)
label_image = sitk.ReadImage(label_path)

print("Case ID:", case_id)
print("CT size (X, Y, Z):", ct_image.GetSize())
print("Label size (X, Y, Z):", label_image.GetSize())
print("CT spacing (mm):", ct_image.GetSpacing())
print("Label spacing (mm):", label_image.GetSpacing())
print("CT origin:", ct_image.GetOrigin())
print("CT direction:", ct_image.GetDirection())
print("Label direction:", label_image.GetDirection())

In [ ]:
import SimpleITK as sitk

for case_id in train_ids[:5]:
    # Load the case and write it to the temporary files
    ct_array, label_array = load_case(case_id)

    ct_image = sitk.ReadImage(
        os.path.join(TEMP_DIR, "ct.mha")
    )
    label_image = sitk.ReadImage(
        os.path.join(TEMP_DIR, "label.mha")
    )

    print(f"\nCase: {case_id}")
    print("CT spacing:", ct_image.GetSpacing())
    print("CT direction:", ct_image.GetDirection())
    print("Label matches CT geometry:",
          ct_image.GetSize() == label_image.GetSize()
          and ct_image.GetSpacing() == label_image.GetSpacing()
          and ct_image.GetOrigin() == label_image.GetOrigin()
          and ct_image.GetDirection() == label_image.GetDirection())

In [ ]:
import SimpleITK as sitk
import numpy as np
import os

case_id = "043"
ct_array, label_array = load_case(case_id)

ct_image = sitk.ReadImage(
    os.path.join(TEMP_DIR, "ct.mha")
)
label_image = sitk.ReadImage(
    os.path.join(TEMP_DIR, "label.mha")
)

target_spacing = (1.0, 1.0, 1.0)

def resample_image(image, target_spacing, is_label=False):
    original_spacing = image.GetSpacing()
    original_size = image.GetSize()

    new_size = [
        int(round(original_size[i] * original_spacing[i]
                  / target_spacing[i]))
        for i in range(3)
    ]

    resampler = sitk.ResampleImageFilter()
    resampler.SetOutputSpacing(target_spacing)
    resampler.SetSize(new_size)
    resampler.SetOutputOrigin(image.GetOrigin())
    resampler.SetOutputDirection(image.GetDirection())
    resampler.SetTransform(sitk.Transform())
    resampler.SetDefaultPixelValue(0)

    if is_label:
        resampler.SetInterpolator(sitk.sitkNearestNeighbor)
    else:
        resampler.SetInterpolator(sitk.sitkLinear)

    return resampler.Execute(image)

ct_resampled = resample_image(
    ct_image, target_spacing, is_label=False
)
label_resampled = resample_image(
    label_image, target_spacing, is_label=True
)

ct_resampled_array = sitk.GetArrayFromImage(ct_resampled)
label_resampled_array = sitk.GetArrayFromImage(label_resampled)

print("Original CT shape:", ct_array.shape)
print("Resampled CT shape:", ct_resampled_array.shape)
print("Resampled label shape:", label_resampled_array.shape)
print("Target spacing:", ct_resampled.GetSpacing())
print("Labels preserved:", np.array_equal(
    np.unique(label_array),
    np.unique(label_resampled_array)
))
print("CT-label shapes match:",
      ct_resampled_array.shape == label_resampled_array.shape)

In [ ]:
import numpy as np

original_labels, original_counts = np.unique(
    label_array, return_counts=True
)
resampled_labels, resampled_counts = np.unique(
    label_resampled_array, return_counts=True
)

print("Original fragment voxel counts:")
for label_id, count in zip(original_labels, original_counts):
    if label_id != 0:
        print(f"Label {label_id}: {count:,}")

print("\nResampled fragment voxel counts:")
for label_id, count in zip(resampled_labels, resampled_counts):
    if label_id != 0:
        print(f"Label {label_id}: {count:,}")

print("\nOriginal unique labels:", len(original_labels))
print("Resampled unique labels:", len(resampled_labels))
print("All original labels preserved:",
      set(original_labels).issubset(set(resampled_labels)))

In [ ]:
import numpy as np
import SimpleITK as sitk
import os

def preprocess_case(case_id, target_spacing=(1.0, 1.0, 1.0)):
    # Load CT and matching label
    load_case(case_id)

    ct_image = sitk.ReadImage(os.path.join(TEMP_DIR, "ct.mha"))
    label_image = sitk.ReadImage(os.path.join(TEMP_DIR, "label.mha"))

    # Check original geometry
    assert ct_image.GetSize() == label_image.GetSize()
    assert ct_image.GetSpacing() == label_image.GetSpacing()
    assert ct_image.GetOrigin() == label_image.GetOrigin()
    assert ct_image.GetDirection() == label_image.GetDirection()

    # Resample CT using linear interpolation
    ct_resampled = resample_image(
        ct_image, target_spacing, is_label=False
    )

    # Resample labels using nearest-neighbor interpolation
    label_resampled = resample_image(
        label_image, target_spacing, is_label=True
    )

    ct = sitk.GetArrayFromImage(ct_resampled).astype(np.float32)
    label = sitk.GetArrayFromImage(label_resampled).astype(np.int16)

    # Normalize CT intensities
    ct = np.clip(ct, -200, 2000)
    ct = (ct + 200.0) / 2200.0

    assert ct.shape == label.shape
    assert np.isfinite(ct).all()

    return ct, label


# Test the complete preprocessing pipeline
processed_ct, processed_label = preprocess_case("043")

print("Preprocessing successful!")
print("CT shape:", processed_ct.shape)
print("Label shape:", processed_label.shape)
print("CT intensity range:",
      float(processed_ct.min()), float(processed_ct.max()))
print("Labels:", np.unique(processed_label))

In [ ]:
import random
import numpy as np

def sample_preprocessed_patch(ct, label, patch_size=96):
    depth, height, width = ct.shape

    if min(depth, height, width) < patch_size:
        raise ValueError("Volume is smaller than the requested patch.")

    # Choose a random labeled bone voxel
    coords = np.argwhere(label > 0)

    if len(coords) == 0:
        raise ValueError("No labeled bone fragments found.")

    z, y, x = coords[random.randrange(len(coords))]

    # Choose patch start coordinates
    def start_index(center, dimension):
        return max(0, min(
            center - patch_size // 2,
            dimension - patch_size
        ))

    z0 = start_index(z, depth)
    y0 = start_index(y, height)
    x0 = start_index(x, width)

    ct_patch = ct[
        z0:z0 + patch_size,
        y0:y0 + patch_size,
        x0:x0 + patch_size
    ]

    label_patch = label[
        z0:z0 + patch_size,
        y0:y0 + patch_size,
        x0:x0 + patch_size
    ]

    return ct_patch.copy(), label_patch.copy()


# Test on preprocessed case 043
ct_patch, label_patch = sample_preprocessed_patch(
    processed_ct, processed_label
)

print("CT patch:", ct_patch.shape)
print("Label patch:", label_patch.shape)
print("Intensity range:",
      float(ct_patch.min()), float(ct_patch.max()))
print("Labels present:", np.unique(label_patch))
print("Preprocessed patch sampler works!")

In [ ]:
import torch
import numpy as np
import gc

# Reinitialize model for a clean experiment
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=31,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1,
    norm="INSTANCE"
).to(device)

loss_function = DiceCELoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

model.train()
losses = []

# Small pilot: 5 cases, 2 patches per case
for case_id in train_ids[:5]:
    print("Processing case:", case_id)

    ct, label = preprocess_case(case_id)

    for _ in range(2):
        ct_patch, label_patch = sample_preprocessed_patch(
            ct, label, patch_size=96
        )

        x = torch.from_numpy(
            ct_patch.copy()
        ).unsqueeze(0).unsqueeze(0).float().to(device)

        y = torch.from_numpy(
            label_patch.copy()
        ).unsqueeze(0).long().to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_function(logits, y.unsqueeze(1))

        if not torch.isfinite(loss):
            raise RuntimeError(f"Invalid loss for case {case_id}")

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        del x, y, logits, loss

    del ct, label, ct_patch, label_patch
    gc.collect()
    torch.cuda.empty_cache()

print("\nClean preprocessing pilot completed.")
print("Training steps:", len(losses))
print("Mean training loss:", round(float(np.mean(losses)), 4))

In [ ]:
import os
import gc
import torch
import numpy as np

CHECKPOINT_DIR = os.path.join(BASE, "checkpoints")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

model.train()
epoch_losses = []

for i, case_id in enumerate(train_ids, start=1):
    print(f"Case {i}/{len(train_ids)}: {case_id}")

    ct, label = preprocess_case(case_id)

    for _ in range(2):
        ct_patch, label_patch = sample_preprocessed_patch(
            ct, label, patch_size=96
        )

        x = torch.from_numpy(
            ct_patch.copy()
        ).unsqueeze(0).unsqueeze(0).float().to(device)

        y = torch.from_numpy(
            label_patch.copy()
        ).unsqueeze(0).long().to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_function(logits, y.unsqueeze(1))

        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite loss: case {case_id}")

        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

        del x, y, logits, loss

    del ct, label, ct_patch, label_patch
    gc.collect()
    torch.cuda.empty_cache()

    if i % 10 == 0:
        print(
            f"Processed {i} cases | "
            f"Mean loss: {np.mean(epoch_losses):.4f}"
        )

checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_resampled_epoch1.pth"
)

torch.save({
    "epoch": 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "mean_training_loss": float(np.mean(epoch_losses)),
    "train_ids": train_ids,
    "val_ids": val_ids,
    "test_ids": test_ids,
    "target_spacing": (1.0, 1.0, 1.0),
    "patch_size": 96
}, checkpoint_path)

print("\nTraining epoch completed.")
print("Training steps:", len(epoch_losses))
print("Mean training loss:", round(float(np.mean(epoch_losses)), 4))
print("Checkpoint saved:", checkpoint_path)

In [ ]:
import torch
import numpy as np
import gc

model.eval()
validation_dice = []

for i, case_id in enumerate(val_ids, start=1):
    print(f"Validating case {i}/{len(val_ids)}: {case_id}")

    # Apply the same preprocessing used for training
    ct, label = preprocess_case(case_id)

    # Sample one patch containing a bone fragment
    ct_patch, label_patch = sample_preprocessed_patch(
        ct, label, patch_size=96
    )

    x = torch.from_numpy(
        ct_patch.copy()
    ).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        prediction = model(x).argmax(dim=1).squeeze(0)
        prediction = prediction.cpu().numpy()

    # Foreground Dice: all bone fragments combined
    pred_fg = prediction > 0
    gt_fg = label_patch > 0

    denominator = pred_fg.sum() + gt_fg.sum()
    if denominator == 0:
        dice = 1.0
    else:
        intersection = np.logical_and(pred_fg, gt_fg).sum()
        dice = 2.0 * intersection / denominator

    validation_dice.append(float(dice))
    print(f"Dice: {dice:.4f}")

    del ct, label, ct_patch, label_patch, x, prediction
    gc.collect()
    torch.cuda.empty_cache()

print("\nMean validation Dice:",
      round(float(np.mean(validation_dice)), 4))
print("Cases evaluated:", len(validation_dice))

In [ ]:
import torch
import numpy as np
import gc

model.eval()
validation_dice = []

for i, case_id in enumerate(val_ids, start=1):
    print(f"Validating case {i}/{len(val_ids)}: {case_id}")

    # Apply the same preprocessing used for training
    ct, label = preprocess_case(case_id)

    # Sample one patch containing a bone fragment
    ct_patch, label_patch = sample_preprocessed_patch(
        ct, label, patch_size=96
    )

    x = torch.from_numpy(
        ct_patch.copy()
    ).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        prediction = model(x).argmax(dim=1).squeeze(0)
        prediction = prediction.cpu().numpy()

    # Foreground Dice: all bone fragments combined
    pred_fg = prediction > 0
    gt_fg = label_patch > 0

    denominator = pred_fg.sum() + gt_fg.sum()
    if denominator == 0:
        dice = 1.0
    else:
        intersection = np.logical_and(pred_fg, gt_fg).sum()
        dice = 2.0 * intersection / denominator

    validation_dice.append(float(dice))
    print(f"Dice: {dice:.4f}")

    del ct, label, ct_patch, label_patch, x, prediction
    gc.collect()
    torch.cuda.empty_cache()

print("\nMean validation Dice:",
      round(float(np.mean(validation_dice)), 4))
print("Cases evaluated:", len(validation_dice))

In [ ]:
import torch
import numpy as np
import gc

model.eval()
validation_dice = []

for i, case_id in enumerate(val_ids, start=1):
    print(f"Validating case {i}/{len(val_ids)}: {case_id}")

    # Apply the same preprocessing used for training
    ct, label = preprocess_case(case_id)

    # Sample one patch containing a bone fragment
    ct_patch, label_patch = sample_preprocessed_patch(
        ct, label, patch_size=96
    )

    x = torch.from_numpy(
        ct_patch.copy()
    ).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        prediction = model(x).argmax(dim=1).squeeze(0)
        prediction = prediction.cpu().numpy()

    # Foreground Dice: all bone fragments combined
    pred_fg = prediction > 0
    gt_fg = label_patch > 0

    denominator = pred_fg.sum() + gt_fg.sum()
    if denominator == 0:
        dice = 1.0
    else:
        intersection = np.logical_and(pred_fg, gt_fg).sum()
        dice = 2.0 * intersection / denominator

    validation_dice.append(float(dice))
    print(f"Dice: {dice:.4f}")

    del ct, label, ct_patch, label_patch, x, prediction
    gc.collect()
    torch.cuda.empty_cache()

print("\nMean validation Dice:",
      round(float(np.mean(validation_dice)), 4))
print("Cases evaluated:", len(validation_dice))

In [ ]:
import os
import json
import numpy as np

validation_results = {
    "experiment": "baseline_resampled_epoch1",
    "target_spacing_mm": [1.0, 1.0, 1.0],
    "patch_size": 96,
    "patches_per_validation_case": 1,
    "metric": "foreground Dice",
    "mean_validation_dice": float(np.mean(validation_dice)),
    "per_case_dice": {
        case_id: float(score)
        for case_id, score in zip(val_ids, validation_dice)
    }
}

results_path = os.path.join(
    BASE, "baseline_resampled_validation_epoch1.json"
)

with open(results_path, "w") as f:
    json.dump(validation_results, f, indent=2)

print("Validation results saved:", results_path)
print("Mean Dice:", validation_results["mean_validation_dice"])

In [ ]:
import random
import numpy as np

VALIDATION_SEED = 123

random.seed(VALIDATION_SEED)
np.random.seed(VALIDATION_SEED)

print("Validation random seed:", VALIDATION_SEED)
print("Validation cases:", len(val_ids))
print("Fixed validation case IDs:", val_ids)

In [ ]:
import os
import json
import numpy as np

VALIDATION_PATCH_SEED = 123
PATCH_SIZE = 96

rng = np.random.default_rng(VALIDATION_PATCH_SEED)
fixed_validation_patches = {}

for case_id in val_ids:
    # Preprocess using the same pipeline as training
    ct, label = preprocess_case(case_id)

    # Find labeled bone voxels
    coords = np.argwhere(label > 0)

    if len(coords) == 0:
        raise ValueError(f"No bone labels found in case {case_id}")

    # Select a reproducible fragment voxel
    z, y, x = coords[rng.integers(len(coords))]

    def start_index(center, dimension):
        return max(
            0,
            min(center - PATCH_SIZE // 2, dimension - PATCH_SIZE)
        )

    fixed_validation_patches[case_id] = {
        "z": start_index(int(z), ct.shape[0]),
        "y": start_index(int(y), ct.shape[1]),
        "x": start_index(int(x), ct.shape[2]),
        "patch_size": PATCH_SIZE
    }

    print(f"Saved coordinates for case {case_id}")

    del ct, label

patch_path = os.path.join(
    BASE, "fixed_validation_patches.json"
)

with open(patch_path, "w") as f:
    json.dump(fixed_validation_patches, f, indent=2)

print("\nFixed validation patches saved:", patch_path)
print("Total fixed patches:", len(fixed_validation_patches))

In [ ]:
import os
import json
import numpy as np
import torch
import gc

# Load saved validation coordinates
patch_path = os.path.join(
    BASE, "fixed_validation_patches.json"
)

with open(patch_path, "r") as f:
    fixed_patches = json.load(f)

model.eval()
fixed_dice_scores = []

for case_id in val_ids:
    print("Evaluating case:", case_id)

    ct, label = preprocess_case(case_id)
    coords = fixed_patches[case_id]

    z, y, x0 = coords["z"], coords["y"], coords["x"]
    size = coords["patch_size"]

    ct_patch = ct[z:z+size, y:y+size, x0:x0+size]
    label_patch = label[z:z+size, y:y+size, x0:x0+size]

    x_tensor = torch.from_numpy(
        ct_patch.copy()
    ).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        prediction = model(x_tensor).argmax(dim=1).squeeze(0)
        prediction = prediction.cpu().numpy()

    pred_fg = prediction > 0
    gt_fg = label_patch > 0

    denominator = pred_fg.sum() + gt_fg.sum()
    if denominator == 0:
        dice = 1.0
    else:
        intersection = np.logical_and(pred_fg, gt_fg).sum()
        dice = 2.0 * intersection / denominator

    fixed_dice_scores.append(float(dice))
    print(f"Fixed-patch Dice: {dice:.4f}")

    del ct, label, ct_patch, label_patch, x_tensor, prediction
    gc.collect()
    torch.cuda.empty_cache()

mean_fixed_dice = float(np.mean(fixed_dice_scores))

print("\nFixed-patch mean foreground Dice:",
      round(mean_fixed_dice, 4))
print("Validation cases:", len(fixed_dice_scores))

In [ ]:
import os
import gc
import torch
import numpy as np

model.train()
epoch2_losses = []

for i, case_id in enumerate(train_ids, start=1):
    print(f"Epoch 2 — case {i}/70: {case_id}")

    ct, label = preprocess_case(case_id)

    for _ in range(2):
        ct_patch, label_patch = sample_preprocessed_patch(
            ct, label, patch_size=96
        )

        x = torch.from_numpy(
            ct_patch.copy()
        ).unsqueeze(0).unsqueeze(0).float().to(device)

        y = torch.from_numpy(
            label_patch.copy()
        ).unsqueeze(0).long().to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_function(logits, y.unsqueeze(1))

        if not torch.isfinite(loss):
            raise RuntimeError(f"Invalid loss for case {case_id}")

        loss.backward()
        optimizer.step()
        epoch2_losses.append(loss.item())

        del x, y, logits, loss

    del ct, label, ct_patch, label_patch
    gc.collect()
    torch.cuda.empty_cache()

    if i % 10 == 0:
        print(
            f"Processed {i}/70 cases | "
            f"Mean loss: {np.mean(epoch2_losses):.4f}"
        )

checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_resampled_epoch2.pth"
)

torch.save({
    "epoch": 2,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "mean_training_loss": float(np.mean(epoch2_losses)),
    "train_ids": train_ids,
    "val_ids": val_ids,
    "test_ids": test_ids,
    "target_spacing": (1.0, 1.0, 1.0),
    "patch_size": 96
}, checkpoint_path)

print("\nEpoch 2 completed.")
print("Training steps:", len(epoch2_losses))
print("Mean training loss:", round(float(np.mean(epoch2_losses)), 4))
print("Checkpoint:", checkpoint_path)

In [ ]:
import torch
import numpy as np
import gc

model.eval()
epoch2_dice = []

for case_id in val_ids:
    ct, label = preprocess_case(case_id)
    coords = fixed_patches[case_id]

    z, y, x0 = coords["z"], coords["y"], coords["x"]
    size = coords["patch_size"]

    ct_patch = ct[z:z+size, y:y+size, x0:x0+size]
    label_patch = label[z:z+size, y:y+size, x0:x0+size]

    x_tensor = torch.from_numpy(
        ct_patch.copy()
    ).unsqueeze(0).unsqueeze(0).float().to(device)

    with torch.no_grad():
        prediction = model(x_tensor).argmax(dim=1).squeeze(0)
        prediction = prediction.cpu().numpy()

    pred_fg = prediction > 0
    gt_fg = label_patch > 0
    denominator = pred_fg.sum() + gt_fg.sum()

    if denominator == 0:
        dice = 1.0
    else:
        intersection = np.logical_and(pred_fg, gt_fg).sum()
        dice = 2.0 * intersection / denominator

    epoch2_dice.append(float(dice))
    print(f"Case {case_id}: Dice = {dice:.4f}")

    del ct, label, ct_patch, label_patch, x_tensor, prediction
    gc.collect()
    torch.cuda.empty_cache()

print("\nEpoch 2 mean fixed-patch Dice:",
      round(float(np.mean(epoch2_dice)), 4))

In [ ]:
import os
import gc
import torch
import numpy as np

model.train()
epoch3_losses = []

for i, case_id in enumerate(train_ids, start=1):
    print(f"Epoch 3 — case {i}/70: {case_id}")

    ct, label = preprocess_case(case_id)

    for _ in range(2):
        ct_patch, label_patch = sample_preprocessed_patch(
            ct, label, patch_size=96
        )

        x = torch.from_numpy(
            ct_patch.copy()
        ).unsqueeze(0).unsqueeze(0).float().to(device)

        y = torch.from_numpy(
            label_patch.copy()
        ).unsqueeze(0).long().to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = loss_function(logits, y.unsqueeze(1))

        if not torch.isfinite(loss):
            raise RuntimeError(f"Invalid loss for case {case_id}")

        loss.backward()
        optimizer.step()
        epoch3_losses.append(loss.item())

        del x, y, logits, loss

    del ct, label, ct_patch, label_patch
    gc.collect()
    torch.cuda.empty_cache()

    if i % 10 == 0:
        print(
            f"Processed {i}/70 | "
            f"Mean loss: {np.mean(epoch3_losses):.4f}"
        )

checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_resampled_epoch3.pth"
)

torch.save({
    "epoch": 3,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "mean_training_loss": float(np.mean(epoch3_losses)),
    "train_ids": train_ids,
    "val_ids": val_ids,
    "test_ids": test_ids,
    "target_spacing": (1.0, 1.0, 1.0),
    "patch_size": 96
}, checkpoint_path)

print("\nEpoch 3 completed.")
print("Training steps:", len(epoch3_losses))
print("Mean training loss:", round(float(np.mean(epoch3_losses)), 4))
print("Checkpoint saved:", checkpoint_path)

In [ ]:
import os
import gc
import json
import torch
import numpy as np

NUM_EPOCHS = 10
PATCHES_PER_CASE = 2

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

history = []
best_dice = -1.0
best_checkpoint = os.path.join(
    CHECKPOINT_DIR, "baseline_best.pth"
)

for epoch in range(1, NUM_EPOCHS + 1):
    # ---------------- TRAINING ----------------
    model.train()
    train_losses = []

    print(f"\n========== Training epoch {epoch}/{NUM_EPOCHS} ==========")

    for i, case_id in enumerate(train_ids, start=1):
        ct, label = preprocess_case(case_id)

        for _ in range(PATCHES_PER_CASE):
            ct_patch, label_patch = sample_preprocessed_patch(
                ct, label, patch_size=96
            )

            x = torch.from_numpy(
                ct_patch.copy()
            ).unsqueeze(0).unsqueeze(0).float().to(device)

            y = torch.from_numpy(
                label_patch.copy()
            ).unsqueeze(0).long().to(device)

            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_function(logits, y.unsqueeze(1))

            if not torch.isfinite(loss):
                raise RuntimeError(
                    f"Invalid loss at epoch {epoch}, case {case_id}"
                )

            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

            del x, y, logits, loss

        del ct, label
        gc.collect()
        torch.cuda.empty_cache()

        if i % 10 == 0:
            print(
                f"Cases {i}/70 | "
                f"Training loss: {np.mean(train_losses):.4f}"
            )

    mean_train_loss = float(np.mean(train_losses))

    # ---------------- VALIDATION ----------------
    model.eval()
    dice_scores = []

    for case_id in val_ids:
        ct, label = preprocess_case(case_id)
        coords = fixed_patches[case_id]

        z, y0, x0 = coords["z"], coords["y"], coords["x"]
        size = coords["patch_size"]

        ct_patch = ct[z:z+size, y0:y0+size, x0:x0+size]
        label_patch = label[z:z+size, y0:y0+size, x0:x0+size]

        x_tensor = torch.from_numpy(
            ct_patch.copy()
        ).unsqueeze(0).unsqueeze(0).float().to(device)

        with torch.no_grad():
            prediction = model(x_tensor).argmax(dim=1).squeeze(0)
            prediction = prediction.cpu().numpy()

        pred_fg = prediction > 0
        gt_fg = label_patch > 0
        denominator = pred_fg.sum() + gt_fg.sum()

        if denominator == 0:
            dice = 1.0
        else:
            intersection = np.logical_and(pred_fg, gt_fg).sum()
            dice = 2.0 * intersection / denominator

        dice_scores.append(float(dice))

        del ct, label, ct_patch, label_patch, x_tensor, prediction
        gc.collect()
        torch.cuda.empty_cache()

    mean_val_dice = float(np.mean(dice_scores))

    record = {
        "epoch": epoch,
        "train_loss": mean_train_loss,
        "validation_foreground_dice": mean_val_dice
    }
    history.append(record)

    # Save checkpoint after each epoch
    checkpoint = {
        "epoch": epoch + 3,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "train_loss": mean_train_loss,
        "validation_dice": mean_val_dice,
        "train_ids": train_ids,
        "val_ids": val_ids,
        "test_ids": test_ids,
        "target_spacing": (1.0, 1.0, 1.0),
        "patch_size": 96
    }

    torch.save(
        checkpoint,
        os.path.join(
            CHECKPOINT_DIR, f"baseline_continued_epoch{epoch + 3}.pth"
        )
    )

    if mean_val_dice > best_dice:
        best_dice = mean_val_dice
        torch.save(checkpoint, best_checkpoint)

    print(
        f"Epoch {epoch + 3} summary | "
        f"Train loss: {mean_train_loss:.4f} | "
        f"Validation Dice: {mean_val_dice:.4f} | "
        f"Best Dice: {best_dice:.4f}"
    )

    with open(
        os.path.join(BASE, "baseline_training_history.json"), "w"
    ) as f:
        json.dump(history, f, indent=2)

print("\nTraining finished.")
print("Best validation Dice:", round(best_dice, 4))
print("Best checkpoint:", best_checkpoint)

In [ ]:
import os
import torch

best_checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_best.pth"
)

checkpoint = torch.load(
    best_checkpoint_path,
    map_location=device,
    weights_only=False
)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

print("Best checkpoint loaded successfully!")
print("Saved epoch:", checkpoint["epoch"])
print("Saved validation Dice:", checkpoint["validation_dice"])
print("Model device:", next(model.parameters()).device)
print("Model is in evaluation mode:", not model.training)

In [ ]:
from monai.inferers import sliding_window_inference

print("Sliding-window inference is ready!")
print("Input patch size: 96 × 96 × 96")
print("Output classes:", 31)
print("Model is in evaluation mode:", not model.training)
print("Model device:", next(model.parameters()).device)

In [ ]:
# Select one validation case
case_id = "001"

# Load and preprocess the CT scan
ct_volume, label_volume = preprocess_case(
    case_id,
    target_spacing=(1.0, 1.0, 1.0)
)

print("Case ID:", case_id)
print("CT volume shape (Z, Y, X):", ct_volume.shape)
print("Label volume shape (Z, Y, X):", label_volume.shape)
print("CT intensity range:",
      float(ct_volume.min()),
      "to",
      float(ct_volume.max()))
print("Unique labels:", __import__("numpy").unique(label_volume))

In [ ]:
import torch
import numpy as np
from monai.inferers import sliding_window_inference

# Convert the CT volume to the model's input format:
# (batch, channel, Z, Y, X)
ct_tensor = torch.from_numpy(
    ct_volume.astype(np.float32)
).unsqueeze(0).unsqueeze(0).to(device)

print("Input tensor shape:", tuple(ct_tensor.shape))

# Predict the complete volume using overlapping patches
with torch.no_grad():
    logits = sliding_window_inference(
        inputs=ct_tensor,
        roi_size=(96, 96, 96),
        sw_batch_size=1,
        predictor=model,
        overlap=0.25,
        mode="gaussian"
    )

    # Convert class scores into predicted label IDs
    prediction = torch.argmax(logits, dim=1).squeeze(0)

# Move prediction to CPU for further processing
prediction = prediction.cpu().numpy().astype(np.uint8)

del ct_tensor, logits
torch.cuda.empty_cache()

print("Prediction completed!")
print("Prediction shape:", prediction.shape)
print("Predicted label IDs:", np.unique(prediction))
print("Predicted bone voxels:", int(np.count_nonzero(prediction)))
print("Total voxels:", prediction.size)

In [ ]:
import numpy as np

# Treat every non-background label as bone
pred_bone = prediction > 0
true_bone = label_volume > 0

# Calculate intersection and Dice score
intersection = np.logical_and(pred_bone, true_bone).sum()
pred_count = pred_bone.sum()
true_count = true_bone.sum()

dice = (
    2.0 * intersection / (pred_count + true_count)
    if (pred_count + true_count) > 0
    else 1.0
)

print("Full-volume foreground evaluation")
print("---------------------------------")
print(f"Dice score: {dice:.4f}")
print(f"Predicted bone voxels: {pred_count:,}")
print(f"Ground-truth bone voxels: {true_count:,}")
print(f"Intersection voxels: {intersection:,}")
print(f"Predicted bone volume: {pred_count / 1000:.2f} cm³")
print(f"Ground-truth bone volume: {true_count / 1000:.2f} cm³")
print("Predicted class IDs:", np.unique(prediction))

In [ ]:
import numpy as np

# Treat every non-background label as bone
pred_bone = prediction > 0
true_bone = label_volume > 0

# Calculate intersection and Dice score
intersection = np.logical_and(pred_bone, true_bone).sum()
pred_count = pred_bone.sum()
true_count = true_bone.sum()

dice = (
    2.0 * intersection / (pred_count + true_count)
    if (pred_count + true_count) > 0
    else 1.0
)

print("Full-volume foreground evaluation")
print("---------------------------------")
print(f"Dice score: {dice:.4f}")
print(f"Predicted bone voxels: {pred_count:,}")
print(f"Ground-truth bone voxels: {true_count:,}")
print(f"Intersection voxels: {intersection:,}")
print(f"Predicted bone volume: {pred_count / 1000:.2f} cm³")
print(f"Ground-truth bone volume: {true_count / 1000:.2f} cm³")
print("Predicted class IDs:", np.unique(prediction))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Select the slice containing the most ground-truth bone voxels
bone_per_slice = np.sum(label_volume > 0, axis=(1, 2))
z = int(np.argmax(bone_per_slice))

ct_slice = ct_volume[z]
true_mask = label_volume[z] > 0
pred_mask = prediction[z] > 0

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

axes[0].imshow(ct_slice, cmap="gray")
axes[0].set_title(f"Original CT — Slice {z}")

axes[1].imshow(ct_slice, cmap="gray")
axes[1].contour(true_mask, levels=[0.5], colors=["lime"])
axes[1].set_title("Ground Truth — Green outline")

axes[2].imshow(ct_slice, cmap="gray")
axes[2].contour(true_mask, levels=[0.5], colors=["lime"])
axes[2].contour(pred_mask, levels=[0.5], colors=["red"])
axes[2].set_title("Prediction: Green = True, Red = Predicted")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Visualized slice:", z)
print("Ground-truth bone pixels:", int(true_mask.sum()))
print("Predicted bone pixels:", int(pred_mask.sum()))

In [ ]:
import numpy as np

# Boolean masks for the selected slice
true_mask = label_volume[z] > 0
pred_mask = prediction[z] > 0

# Calculate segmentation outcomes
true_positive = np.logical_and(true_mask, pred_mask).sum()
false_positive = np.logical_and(~true_mask, pred_mask).sum()
false_negative = np.logical_and(true_mask, ~pred_mask).sum()

precision = (
    true_positive / (true_positive + false_positive)
    if true_positive + false_positive > 0 else 0
)

recall = (
    true_positive / (true_positive + false_negative)
    if true_positive + false_negative > 0 else 0
)

print("Slice:", z)
print("True positives:", int(true_positive))
print("False positives:", int(false_positive))
print("False negatives:", int(false_negative))
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")

In [ ]:
import numpy as np

print("Ground-truth label counts:")
for label_id, count in zip(*np.unique(label_volume, return_counts=True)):
    if label_id != 0:
        print(f"Label {label_id}: {count:,} voxels")

print("\nPredicted label counts:")
for label_id, count in zip(*np.unique(prediction, return_counts=True)):
    if label_id != 0:
        print(f"Label {label_id}: {count:,} voxels")

In [ ]:
import numpy as np

print("Class-wise full-volume Dice")
print("-" * 35)

for class_id in np.unique(label_volume):
    if class_id == 0:
        continue

    true_class = label_volume == class_id
    pred_class = prediction == class_id

    intersection = np.logical_and(
        true_class, pred_class
    ).sum()

    denominator = true_class.sum() + pred_class.sum()

    dice = (
        2.0 * intersection / denominator
        if denominator > 0 else 1.0
    )

    print(
        f"Class {class_id}: "
        f"Dice = {dice:.4f} | "
        f"Ground truth = {true_class.sum():,} | "
        f"Predicted = {pred_class.sum():,}"
    )

In [ ]:
import json
import os

history_path = os.path.join(
    BASE, "baseline_training_history.json"
)

print("History file exists:", os.path.exists(history_path))

with open(history_path, "r") as f:
    history = json.load(f)

print("History data type:", type(history).__name__)

if isinstance(history, dict):
    for key, value in history.items():
        print(f"\n{key}:")
        print(value if not isinstance(value, list) else value[-5:])
elif isinstance(history, list):
    print("Last five training records:")
    for record in history[-5:]:
        print(record)

In [ ]:
import os
import torch

checkpoint_path = os.path.join(
    CHECKPOINT_DIR, "baseline_best.pth"
)

checkpoint = torch.load(
    checkpoint_path,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint keys:", list(checkpoint.keys()))

for key in [
    "epoch",
    "train_loss",
    "validation_dice",
    "target_spacing",
    "patch_size",
    "train_ids",
    "val_ids"
]:
    if key in checkpoint:
        value = checkpoint[key]
        if key in ["train_ids", "val_ids"]:
            print(f"{key}: {len(value)} cases")
        else:
            print(f"{key}: {value}")

In [ ]:
import numpy as np

print("Current CT volume:")
print("Shape:", ct_volume.shape)
print("Minimum:", float(ct_volume.min()))
print("Maximum:", float(ct_volume.max()))
print("Mean:", float(ct_volume.mean()))
print("Standard deviation:", float(ct_volume.std()))

print("\nGround-truth foreground fraction:")
print(f"{np.mean(label_volume > 0) * 100:.3f}%")

print("\nPredicted foreground fraction:")
print(f"{np.mean(prediction > 0) * 100:.3f}%")

In [ ]:
import inspect

print(inspect.getsource(preprocess_case))

In [ ]:
import inspect

print(inspect.getsource(sample_preprocessed_patch))

In [ ]:
import numpy as np
import random

case_id = "043"

ct, label = preprocess_case(case_id)

for i in range(5):
    ct_patch, label_patch = sample_preprocessed_patch(
        ct, label, patch_size=96
    )

    foreground_fraction = np.mean(label_patch > 0)

    print(
        f"Patch {i + 1}: "
        f"shape={label_patch.shape}, "
        f"bone fraction={foreground_fraction:.2%}, "
        f"classes={np.unique(label_patch).tolist()}"
    )

In [ ]:
import numpy as np
from collections import Counter

case_id = "043"
ct, label = preprocess_case(case_id)

class_patch_counts = Counter()
class_voxel_counts = Counter()
bone_fractions = []

for i in range(50):
    _, label_patch = sample_preprocessed_patch(
        ct, label, patch_size=96
    )

    bone_fractions.append(np.mean(label_patch > 0))

    present_classes = np.unique(label_patch)
    for class_id in present_classes:
        if class_id != 0:
            class_patch_counts[int(class_id)] += 1

    class_voxel_counts.update(
        label_patch[label_patch > 0].astype(int).tolist()
    )

print("Average bone fraction:",
      f"{np.mean(bone_fractions):.2%}")

print("\nNumber of patches containing each class:")
for class_id, count in sorted(class_patch_counts.items()):
    print(f"Class {class_id}: {count}/50 patches")

print("\nBone voxel counts across sampled patches:")
for class_id, count in sorted(class_voxel_counts.items()):
    print(f"Class {class_id}: {count:,}")

In [ ]:
from collections import Counter
import numpy as np

training_cases = ["043", "042", "092", "010", "066"]
global_class_counts = Counter()

for case_id in training_cases:
    _, labels = preprocess_case(case_id)

    counts = Counter(
        labels[labels > 0].astype(int).tolist()
    )
    global_class_counts.update(counts)

    print(f"\nCase {case_id}")
    for class_id, count in sorted(counts.items()):
        print(f"  Class {class_id}: {count:,} voxels")

print("\nCombined counts across these five training cases:")
for class_id, count in sorted(global_class_counts.items()):
    print(f"Class {class_id}: {count:,} voxels")

In [ ]:
import numpy as np
import random

def sample_class_aware_patch(ct, label, patch_size=96):
    depth, height, width = ct.shape

    if min(depth, height, width) < patch_size:
        raise ValueError("Volume is smaller than the requested patch.")

    # Find the classes actually present in this case
    classes = np.unique(label)
    classes = classes[classes > 0]

    if len(classes) == 0:
        raise ValueError("No labeled bone fragments found.")

    # Choose a class uniformly rather than choosing a voxel
    # uniformly across all bone classes
    class_id = int(random.choice(classes.tolist()))

    coords = np.argwhere(label == class_id)
    z, y, x = coords[random.randrange(len(coords))]

    def start_index(center, dimension):
        return max(
            0,
            min(center - patch_size // 2, dimension - patch_size)
        )

    z0 = start_index(z, depth)
    y0 = start_index(y, height)
    x0 = start_index(x, width)

    ct_patch = ct[
        z0:z0 + patch_size,
        y0:y0 + patch_size,
        x0:x0 + patch_size
    ]

    label_patch = label[
        z0:z0 + patch_size,
        y0:y0 + patch_size,
        x0:x0 + patch_size
    ]

    return ct_patch.copy(), label_patch.copy()

In [ ]:
for i in range(10):
    ct_patch, label_patch = sample_class_aware_patch(
        ct, label, patch_size=96
    )

    print(
        f"Patch {i + 1}: "
        f"bone fraction={np.mean(label_patch > 0):.2%}, "
        f"classes={np.unique(label_patch).tolist()}"
    )

In [ ]:
import os
import torch
from monai.networks.nets import UNet

# Separate checkpoint for the improved experiment
IMPROVED_CHECKPOINT_DIR = os.path.join(
    BASE, "checkpoints_improved"
)
os.makedirs(IMPROVED_CHECKPOINT_DIR, exist_ok=True)

IMPROVED_CHECKPOINT_PATH = os.path.join(
    IMPROVED_CHECKPOINT_DIR, "class_aware_best.pth"
)

# Create a fresh model with the same architecture
improved_model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=31,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1,
    norm="INSTANCE"
).to(device)

# Use a separate optimizer
improved_optimizer = torch.optim.AdamW(
    improved_model.parameters(),
    lr=2e-4,
    weight_decay=1e-5
)

# Use the same loss configuration
improved_loss_function = loss_function

print("Improved model initialized successfully!")
print("Device:", next(improved_model.parameters()).device)
print("Parameters:", sum(p.numel() for p in improved_model.parameters()))
print("Baseline checkpoint preserved:", os.path.exists(
    os.path.join(CHECKPOINT_DIR, "baseline_best.pth")
))
print("New checkpoint path:", IMPROVED_CHECKPOINT_PATH)

In [ ]:
import numpy as np
import torch
import random

# Use training cases only; validation and test cases remain separate
pilot_train_ids = ["043", "042", "092", "010", "066"]

improved_model.train()

pilot_losses = []

for case_id in pilot_train_ids:
    print("Processing training case:", case_id)

    ct, label = preprocess_case(case_id)

    # Two class-aware patches per case for the pilot
    for patch_num in range(2):
        ct_patch, label_patch = sample_class_aware_patch(
            ct, label, patch_size=96
        )

        # Convert to tensors: (batch, channel, Z, Y, X)
        ct_tensor = torch.from_numpy(
            ct_patch.astype(np.float32)
        ).unsqueeze(0).unsqueeze(0).to(device)

        label_tensor = torch.from_numpy(
            label_patch.astype(np.int64)
        ).unsqueeze(0).unsqueeze(0).to(device)

        improved_optimizer.zero_grad(set_to_none=True)

        logits = improved_model(ct_tensor)
        loss = improved_loss_function(logits, label_tensor)

        loss.backward()
        improved_optimizer.step()

        pilot_losses.append(loss.item())

        print(
            f"  Patch {patch_num + 1}/2 — "
            f"loss: {loss.item():.4f}"
        )

        del ct_tensor, label_tensor, logits, loss

    del ct, label
    torch.cuda.empty_cache()

print("\nPilot training completed.")
print("Mean pilot loss:", round(float(np.mean(pilot_losses)), 4))

In [ ]:
import os
import torch

pilot_checkpoint_path = os.path.join(
    IMPROVED_CHECKPOINT_DIR,
    "class_aware_pilot.pth"
)

torch.save(
    {
        "model_state_dict": improved_model.state_dict(),
        "optimizer_state_dict": improved_optimizer.state_dict(),
        "train_loss": float(np.mean(pilot_losses)),
        "train_ids": pilot_train_ids,
        "patch_size": 96,
        "target_spacing": (1.0, 1.0, 1.0)
    },
    pilot_checkpoint_path
)

print("Pilot checkpoint saved!")
print("Path:", pilot_checkpoint_path)
print("Baseline checkpoint remains unchanged:",
      os.path.exists(
          os.path.join(CHECKPOINT_DIR, "baseline_best.pth")
      ))

In [ ]:
import numpy as np
import torch

improved_model.eval()

dice_scores = []

with torch.no_grad():
    for case_id, patch_info in fixed_patches.items():

        # Load the same fixed validation patch
        ct, label = preprocess_case(case_id)

        z0 = patch_info["z"]
        y0 = patch_info["y"]
        x0 = patch_info["x"]
        patch_size = 96

        ct_patch = ct[
            z0:z0 + patch_size,
            y0:y0 + patch_size,
            x0:x0 + patch_size
        ]

        label_patch = label[
            z0:z0 + patch_size,
            y0:y0 + patch_size,
            x0:x0 + patch_size
        ]

        ct_tensor = torch.from_numpy(
            ct_patch.astype(np.float32)
        ).unsqueeze(0).unsqueeze(0).to(device)

        logits = improved_model(ct_tensor)
        prediction_patch = torch.argmax(
            logits, dim=1
        ).squeeze(0).cpu().numpy()

        true_mask = label_patch > 0
        pred_mask = prediction_patch > 0

        intersection = np.logical_and(
            true_mask, pred_mask
        ).sum()

        denominator = true_mask.sum() + pred_mask.sum()

        dice = (
            2.0 * intersection / denominator
            if denominator > 0 else 1.0
        )

        dice_scores.append(float(dice))

        print(f"Case {case_id}: Dice = {dice:.4f}")

        del ct_tensor, logits, ct, label

print("\nImproved pilot mean validation Dice:",
      round(float(np.mean(dice_scores)), 4))

In [ ]:
import os
import torch
from monai.networks.nets import UNet

# Create a separate experiment directory
FINETUNE_DIR = os.path.join(BASE, "checkpoints_class_aware_finetune")
os.makedirs(FINETUNE_DIR, exist_ok=True)

FINETUNE_PATH = os.path.join(
    FINETUNE_DIR, "class_aware_finetuned_best.pth"
)

# Load the original trained baseline
baseline_checkpoint = torch.load(
    os.path.join(CHECKPOINT_DIR, "baseline_best.pth"),
    map_location="cpu",
    weights_only=False
)

# Create a separate model with the same architecture
finetune_model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=31,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1,
    norm="INSTANCE"
).to(device)

# Start from the trained baseline weights
finetune_model.load_state_dict(
    baseline_checkpoint["model_state_dict"]
)

# Separate optimizer with a smaller learning rate
finetune_optimizer = torch.optim.AdamW(
    finetune_model.parameters(),
    lr=5e-5,
    weight_decay=1e-5
)

finetune_loss_function = loss_function

print("Baseline weights loaded into a separate model.")
print("Starting checkpoint epoch:", baseline_checkpoint["epoch"])
print("Learning rate:", 5e-5)
print("Original baseline checkpoint preserved:", os.path.exists(
    os.path.join(CHECKPOINT_DIR, "baseline_best.pth")
))
print("Fine-tuning checkpoint path:", FINETUNE_PATH)

In [ ]:
import numpy as np
import torch

finetune_model.train()
finetune_losses = []

finetune_train_ids = ["043", "042", "092", "010", "066"]

for case_id in finetune_train_ids:
    print("Processing:", case_id)

    ct, label = preprocess_case(case_id)

    for patch_num in range(10):
        ct_patch, label_patch = sample_class_aware_patch(
            ct, label, patch_size=96
        )

        ct_tensor = torch.from_numpy(
            ct_patch.astype(np.float32)
        ).unsqueeze(0).unsqueeze(0).to(device)

        label_tensor = torch.from_numpy(
            label_patch.astype(np.int64)
        ).unsqueeze(0).unsqueeze(0).to(device)

        finetune_optimizer.zero_grad(set_to_none=True)

        logits = finetune_model(ct_tensor)
        loss = finetune_loss_function(logits, label_tensor)

        loss.backward()
        finetune_optimizer.step()

        finetune_losses.append(float(loss.item()))

        if (patch_num + 1) % 5 == 0:
            print(
                f"  Patches {patch_num + 1}/10, "
                f"latest loss: {loss.item():.4f}"
            )

        del ct_tensor, label_tensor, logits, loss

    del ct, label
    torch.cuda.empty_cache()

print("\nFine-tuning pilot completed.")
print("Optimization steps:", len(finetune_losses))
print("Mean training loss:", round(float(np.mean(finetune_losses)), 4))

In [ ]:
import os
import torch
import numpy as np

torch.save(
    {
        "model_state_dict": finetune_model.state_dict(),
        "optimizer_state_dict": finetune_optimizer.state_dict(),
        "train_loss": float(np.mean(finetune_losses)),
        "train_ids": finetune_train_ids,
        "patch_size": 96,
        "target_spacing": (1.0, 1.0, 1.0),
        "starting_checkpoint_epoch": 13
    },
    FINETUNE_PATH
)

print("Fine-tuned checkpoint saved!")
print("Path:", FINETUNE_PATH)
print("Mean training loss:", round(float(np.mean(finetune_losses)), 4))
print("Baseline checkpoint preserved:", os.path.exists(
    os.path.join(CHECKPOINT_DIR, "baseline_best.pth")
))

In [ ]:

import torch
import numpy as np

finetune_model.eval()
dice_scores = []

with torch.no_grad():
    for case_id, patch_info in fixed_patches.items():
        ct, label = preprocess_case(case_id)

        z0 = patch_info["z"]
        y0 = patch_info["y"]
        x0 = patch_info["x"]
        patch_size = 96

        ct_patch = ct[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        label_patch = label[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        ct_tensor = (
            torch.from_numpy(ct_patch.astype(np.float32))
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        logits = finetune_model(ct_tensor)

        prediction_patch = (
            torch.argmax(logits, dim=1)
            .squeeze(0)
            .cpu()
            .numpy()
        )

        true_mask = label_patch > 0
        pred_mask = prediction_patch > 0

        intersection = np.logical_and(
            true_mask, pred_mask
        ).sum()

        denominator = true_mask.sum() + pred_mask.sum()

        dice = (
            2 * intersection / denominator
            if denominator > 0 else 1.0
        )

        dice_scores.append(float(dice))

        print(f"Case {case_id}: Dice = {dice:.4f}")

        del ct_tensor, logits, ct, label

mean_finetune_dice = float(np.mean(dice_scores))

print("\nFine-tuned mean validation Dice:",
      round(mean_finetune_dice, 4))


In [ ]:

import torch
import numpy as np
from monai.inferers import sliding_window_inference

case_id = "001"

# Load the complete validation CT and its ground-truth labels
ct, label = preprocess_case(case_id)

ct_tensor = (
    torch.from_numpy(ct.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device)
)

finetune_model.eval()

with torch.no_grad():
    logits = sliding_window_inference(
        ct_tensor,
        roi_size=(96, 96, 96),
        sw_batch_size=1,
        predictor=finetune_model,
        overlap=0.25,
        mode="gaussian"
    )

prediction = (
    torch.argmax(logits, dim=1)
    .squeeze(0)
    .cpu()
    .numpy()
)

true_mask = label > 0
pred_mask = prediction > 0

intersection = np.logical_and(true_mask, pred_mask).sum()
denominator = true_mask.sum() + pred_mask.sum()

full_volume_dice = (
    2 * intersection / denominator
    if denominator > 0 else 1.0
)

print("Case:", case_id)
print("CT shape:", ct.shape)
print("Ground-truth foreground voxels:", int(true_mask.sum()))
print("Predicted foreground voxels:", int(pred_mask.sum()))
print("Fine-tuned full-volume Dice:", round(float(full_volume_dice), 4))

del ct_tensor, logits, ct, label


In [ ]:

import numpy as np

# Reload the complete case because the previous cell deleted ct and label.
ct, label = preprocess_case("001")

ct_tensor = (
    torch.from_numpy(ct.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device)
)

finetune_model.eval()

with torch.no_grad():
    logits = sliding_window_inference(
        ct_tensor,
        roi_size=(96, 96, 96),
        sw_batch_size=1,
        predictor=finetune_model,
        overlap=0.25,
        mode="gaussian"
    )

prediction = (
    torch.argmax(logits, dim=1)
    .squeeze(0)
    .cpu()
    .numpy()
)

print("Class-wise full-volume Dice")
print("-" * 45)

for class_id in np.unique(label):
    if class_id == 0:
        continue

    true_class = label == class_id
    pred_class = prediction == class_id

    intersection = np.logical_and(
        true_class, pred_class
    ).sum()

    denominator = true_class.sum() + pred_class.sum()

    dice = (
        2 * intersection / denominator
        if denominator > 0 else 1.0
    )

    print(
        f"Class {int(class_id):2d}: "
        f"Dice = {dice:.4f} | "
        f"GT = {int(true_class.sum())} | "
        f"Predicted = {int(pred_class.sum())}"
    )

del ct_tensor, logits, ct, label


In [ ]:

import numpy as np

print("Predicted class distribution")
print("-" * 40)

classes, counts = np.unique(prediction, return_counts=True)

for class_id, count in zip(classes, counts):
    if class_id != 0:
        print(
            f"Class {int(class_id):2d}: "
            f"{int(count):,} voxels "
            f"({100 * count / prediction.size:.3f}%)"
        )

print("\nGround-truth class distribution")
print("-" * 40)

classes, counts = np.unique(label, return_counts=True)

for class_id, count in zip(classes, counts):
    if class_id != 0:
        print(
            f"Class {int(class_id):2d}: "
            f"{int(count):,} voxels "
            f"({100 * count / label.size:.3f}%)"
        )


In [ ]:

ct, label = preprocess_case("001")

print("CT shape:", ct.shape)
print("Label shape:", label.shape)

print("\nGround-truth class distribution")
print("-" * 40)

import numpy as np

classes, counts = np.unique(label, return_counts=True)

for class_id, count in zip(classes, counts):
    if class_id != 0:
        print(
            f"Class {int(class_id):2d}: "
            f"{int(count):,} voxels "
            f"({100 * count / label.size:.3f}%)"
        )


In [ ]:

import inspect
from monai.losses import DiceCELoss

print("Current loss function:")
print(loss_function)

print("\nDiceCELoss supports these parameters:")
print(inspect.signature(DiceCELoss))

print("\nFine-tuning optimizer:")
print(finetune_optimizer)

print("\nFine-tuning training cases:")
print(finetune_train_ids)

print("\nNumber of fine-tuning loss values:",
      len(finetune_losses))

print("Mean fine-tuning training loss:",
      round(float(np.mean(finetune_losses)), 4))


In [ ]:

import numpy as np

check_ids = ["043", "042", "092", "010", "066"]

print("Training-case foreground class distribution")
print("=" * 55)

for case_id in check_ids:
    ct, label = preprocess_case(case_id)

    classes, counts = np.unique(label, return_counts=True)

    print(f"\nCase {case_id}")

    for class_id, count in zip(classes, counts):
        if class_id != 0:
            print(
                f"  Class {int(class_id):2d}: "
                f"{int(count):,} voxels"
            )

    del ct, label

print("\nDiagnostic complete.")


In [ ]:

import numpy as np

def sample_balanced_patch(
    ct,
    label,
    patch_size=96,
    max_attempts=20
):
    """
    Sample a patch centered on a randomly selected
    foreground class, checking class presence in the patch.
    """

    valid_classes = [
        int(c)
        for c in np.unique(label)
        if c != 0 and np.sum(label == c) >= 100
    ]

    if not valid_classes:
        raise ValueError("No usable foreground classes found.")

    for _ in range(max_attempts):
        class_id = int(np.random.choice(valid_classes))

        coords = np.argwhere(label == class_id)
        center = coords[np.random.randint(len(coords))]

        starts = []
        for axis in range(3):
            max_start = max(0, label.shape[axis] - patch_size)
            start = int(center[axis]) - patch_size // 2
            start = max(0, min(start, max_start))
            starts.append(start)

        z0, y0, x0 = starts

        ct_patch = ct[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        label_patch = label[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        # Require the selected class to occupy at least
        # 0.1% of the patch voxels.
        selected_voxels = np.sum(label_patch == class_id)
        minimum_voxels = max(1, int(0.001 * label_patch.size))

        if selected_voxels >= minimum_voxels:
            return ct_patch, label_patch, class_id

    # Safe fallback: return the last patch sampled.
    return ct_patch, label_patch, class_id


print("Balanced patch sampler defined successfully.")


In [ ]:

ct, label = preprocess_case("043")

for i in range(5):
    ct_patch, label_patch, selected_class = sample_balanced_patch(
        ct, label, patch_size=96
    )

    print(
        f"Patch {i+1}: selected class={selected_class}, "
        f"shape={label_patch.shape}, "
        f"selected-class voxels="
        f"{int(np.sum(label_patch == selected_class))}, "
        f"foreground voxels={int(np.sum(label_patch > 0))}"
    )

del ct, label


In [ ]:

import os
import torch
import numpy as np
from monai.networks.nets import UNet
from monai.losses import DiceCELoss

# Keep this experiment separate from all existing checkpoints.
BALANCED_DIR = (
    "/kaggle/working/PENGWIN/checkpoints_balanced_sampling"
)
os.makedirs(BALANCED_DIR, exist_ok=True)

BALANCED_PATH = os.path.join(
    BALANCED_DIR,
    "balanced_sampling_finetuned.pth"
)

# Re-create the same baseline architecture.
balanced_model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=31,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1,
    norm="INSTANCE"
).to(device)

# Load the original baseline weights, not the previous fine-tuned weights.
baseline_checkpoint = torch.load(
    os.path.join(CHECKPOINT_DIR, "baseline_best.pth"),
    map_location=device,
    weights_only=False
)

balanced_model.load_state_dict(
    baseline_checkpoint["model_state_dict"]
)

balanced_optimizer = torch.optim.AdamW(
    balanced_model.parameters(),
    lr=5e-5,
    weight_decay=1e-5
)

balanced_loss_function = DiceCELoss(
    include_background=False,
    to_onehot_y=True,
    softmax=True
)

print("Balanced-sampling experiment initialized.")
print("Starting checkpoint: original baseline")
print("Baseline checkpoint preserved:",
      os.path.exists(
          os.path.join(CHECKPOINT_DIR, "baseline_best.pth")
      ))
print("New checkpoint path:", BALANCED_PATH)


In [ ]:

balanced_model.train()

balanced_train_ids = ["043", "042", "092", "010", "066"]
balanced_losses = []

for case_id in balanced_train_ids:
    print(f"\nTraining on case {case_id}")

    ct, label = preprocess_case(case_id)

    for step in range(10):
        ct_patch, label_patch, selected_class = sample_balanced_patch(
            ct, label, patch_size=96
        )

        ct_tensor = (
            torch.from_numpy(ct_patch.astype(np.float32))
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        label_tensor = (
            torch.from_numpy(label_patch.astype(np.int64))
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        balanced_optimizer.zero_grad(set_to_none=True)

        logits = balanced_model(ct_tensor)
        loss = balanced_loss_function(logits, label_tensor)

        loss.backward()
        balanced_optimizer.step()

        loss_value = float(loss.item())
        balanced_losses.append(loss_value)

        if (step + 1) % 5 == 0:
            print(
                f"Step {step+1}/10 | "
                f"Selected class: {selected_class} | "
                f"Loss: {loss_value:.4f}"
            )

        del ct_tensor, label_tensor, logits, loss

    del ct, label

print("\nBalanced-sampling pilot completed.")
print("Optimization steps:", len(balanced_losses))
print("Mean training loss:", round(float(np.mean(balanced_losses)), 4))


In [ ]:

import os
import torch
import numpy as np

torch.save(
    {
        "model_state_dict": balanced_model.state_dict(),
        "optimizer_state_dict": balanced_optimizer.state_dict(),
        "train_loss": float(np.mean(balanced_losses)),
        "train_ids": balanced_train_ids,
        "patch_size": 96,
        "target_spacing": (1.0, 1.0, 1.0),
        "starting_checkpoint": "baseline_best.pth",
        "optimization_steps": len(balanced_losses)
    },
    BALANCED_PATH
)

print("Balanced-sampling checkpoint saved!")
print("Path:", BALANCED_PATH)
print("Mean training loss:", round(float(np.mean(balanced_losses)), 4))
print(
    "Baseline checkpoint preserved:",
    os.path.exists(
        os.path.join(CHECKPOINT_DIR, "baseline_best.pth")
    )
)


In [ ]:

balanced_model.eval()
balanced_dice_scores = []

with torch.no_grad():
    for case_id, patch_info in fixed_patches.items():
        ct, label = preprocess_case(case_id)

        z0 = patch_info["z"]
        y0 = patch_info["y"]
        x0 = patch_info["x"]
        patch_size = 96

        ct_patch = ct[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        label_patch = label[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        ct_tensor = (
            torch.from_numpy(ct_patch.astype(np.float32))
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        logits = balanced_model(ct_tensor)

        prediction_patch = (
            torch.argmax(logits, dim=1)
            .squeeze(0)
            .cpu()
            .numpy()
        )

        true_mask = label_patch > 0
        pred_mask = prediction_patch > 0

        intersection = np.logical_and(
            true_mask, pred_mask
        ).sum()

        denominator = true_mask.sum() + pred_mask.sum()

        dice = (
            2 * intersection / denominator
            if denominator > 0 else 1.0
        )

        balanced_dice_scores.append(float(dice))

        print(f"Case {case_id}: Dice = {dice:.4f}")

        del ct_tensor, logits, ct, label

balanced_mean_dice = float(np.mean(balanced_dice_scores))

print("\nBalanced-sampling mean validation Dice:",
      round(balanced_mean_dice, 4))


In [ ]:

import torch
import numpy as np
from monai.inferers import sliding_window_inference

ct, label = preprocess_case("001")

ct_tensor = (
    torch.from_numpy(ct.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device)
)

balanced_model.eval()

with torch.no_grad():
    logits = sliding_window_inference(
        ct_tensor,
        roi_size=(96, 96, 96),
        sw_batch_size=1,
        predictor=balanced_model,
        overlap=0.25,
        mode="gaussian"
    )

balanced_prediction = (
    torch.argmax(logits, dim=1)
    .squeeze(0)
    .cpu()
    .numpy()
)

true_mask = label > 0
pred_mask = balanced_prediction > 0

intersection = np.logical_and(
    true_mask, pred_mask
).sum()

denominator = true_mask.sum() + pred_mask.sum()

balanced_full_dice = (
    2 * intersection / denominator
    if denominator > 0 else 1.0
)

print("Case: 001")
print("CT shape:", ct.shape)
print("Ground-truth foreground voxels:", int(true_mask.sum()))
print("Predicted foreground voxels:", int(pred_mask.sum()))
print("Balanced-sampling full-volume Dice:",
      round(float(balanced_full_dice), 4))

del ct_tensor, logits, ct, label


In [ ]:

import numpy as np

ct, label = preprocess_case("001")

ct_tensor = (
    torch.from_numpy(ct.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device)
)

balanced_model.eval()

with torch.no_grad():
    logits = slidi
ng_window_inference(
        ct_tensor,
        roi_size=(96, 96, 96),
        sw_batch_size=1,
        predictor=balanced_model,
        overlap=0.25,
        mode="gaussian"
    )

balanced_prediction = (
    torch.argmax(logits, dim=1)
    .squeeze(0)
    .cpu()
    .numpy()
)

print("Balanced-sampling: class-wise full-volume Dice")
print("-" * 65)

for class_id in np.unique(label):
    if class_id == 0:
        continue

    true_class = label == class_id
    pred_class = balanced_prediction == class_id

    intersection = np.logical_and(
        true_class, pred_class
    ).sum()

    denominator = true_class.sum() + pred_class.sum()

    dice = (
        2 * intersection / denominator
        if denominator > 0 else 1.0
    )

    print(
        f"Class {int(class_id):2d}: "
        f"Dice={dice:.4f} | "
        f"GT={int(true_class.sum()):,} | "
        f"Predicted={int(pred_class.sum()):,}"
    )

del ct_tensor, logits, ct, label


In [ ]:

print("preprocess_case exists:", "preprocess_case" in globals())
print("resample_image exists:", "resample_image" in globals())
print("load_case exists:", "load_case" in globals())
print("balanced_model exists:", "balanced_model" in globals())
print("sliding_window_inference exists:",
      "sliding_window_inference" in globals())


In [ ]:

import os

BASE = "/kaggle/working/PENGWIN"
CHECKPOINT_DIR = os.path.join(BASE, "checkpoints")

paths_to_check = [
    os.path.join(CHECKPOINT_DIR, "baseline_best.pth"),
    os.path.join(
        BASE,
        "checkpoints_class_aware_finetune",
        "class_aware_finetuned_best.pth"
    ),
    os.path.join(
        BASE,
        "checkpoints_balanced_sampling",
        "balanced_sampling_finetuned.pth"
    )
]

for path in paths_to_check:
    print(
        ("FOUND   " if os.path.exists(path) else "MISSING "),
        path
    )


In [ ]:

import os

search_roots = [
    "/kaggle/working",
    "/kaggle/input"
]

found = []

for root in search_roots:
    if not os.path.exists(root):
        continue

    for current_dir, dirs, files in os.walk(root):
        # Avoid searching very large unrelated folders.
        dirs[:] = [
            d for d in dirs
            if d not in [".git", "node_modules"]
        ]

        for filename in files:
            if filename.endswith((".pth", ".pt", ".ckpt")):
                full_path = os.path.join(current_dir, filename)
                found.append(full_path)

print("Checkpoint files found:", len(found))

for path in found[:100]:
    print(path)

if not found:
    print("No checkpoint files found in these locations.")


In [ ]:

import os
import torch

checkpoint_dir = "/kaggle/working/PENGWIN/checkpoints"

for filename in sorted(os.listdir(checkpoint_dir)):
    if filename.endswith(".pth"):
        path = os.path.join(checkpoint_dir, filename)

        print("\nCheckpoint:", filename)
        print("Size:", round(os.path.getsize(path) / (1024 * 1024), 2), "MB")

        try:
            checkpoint = torch.load(
                path,
                map_location="cpu",
                weights_only=False
            )

            if isinstance(checkpoint, dict):
                print("Keys:", list(checkpoint.keys()))

                for key in [
                    "epoch",
                    "train_loss",
                    "validation_dice",
                    "patch_size",
                    "train_ids"
                ]:
                    if key in checkpoint:
                        value = checkpoint[key]
                        if key == "train_ids":
                            value = f"{len(value)} training cases"
                        print(f"{key}: {value}")

        except Exception as e:
            print("Could not inspect:", str(e))


In [ ]:

import os
import torch

BASE = "/kaggle/working/PENGWIN"

checkpoint_files = []

for root, dirs, files in os.walk(BASE):
    for filename in files:
        if filename.endswith(".pth"):
            checkpoint_files.append(
                os.path.join(root, filename)
            )

results = []

for path in checkpoint_files:
    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False
        )

        if not isinstance(ckpt, dict):
            continue

        score = ckpt.get("validation_dice")

        results.append({
            "name": os.path.basename(path),
            "path": path,
            "epoch": ckpt.get("epoch", "N/A"),
            "validation_dice": score,
            "train_loss": ckpt.get("train_loss", "N/A")
        })

    except Exception as e:
        print("Could not read:", path, "|", e)

print("\nCHECKPOINT COMPARISON")
print("=" * 75)

for item in results:
    print("\nName:", item["name"])
    print("Epoch:", item["epoch"])
    print("Validation Dice:", item["validation_dice"])
    print("Training Loss:", item["train_loss"])
    print("Path:", item["path"])

valid = [
    item for item in results
    if item["validation_dice"] is not None
]

if valid:
    best = max(valid, key=lambda x: float(x["validation_dice"]))

    print("\nBEST CHECKPOINT")
    print("Name:", best["name"])
    print("Epoch:", best["epoch"])
    print("Validation Dice:", best["validation_dice"])
    print("Path:", best["path"])


In [ ]:

import torch
from monai.networks.nets import UNet

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CHECKPOINT_PATH = (
    "/kaggle/working/PENGWIN/checkpoints/baseline_best.pth"
)

# Recreate the original 3D U-Net architecture
baseline_model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=31,
    channels=(16, 32, 64, 128),
    strides=(2, 2, 2),
    num_res_units=1,
    norm="INSTANCE"
).to(device)

# Load recovered checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

baseline_model.load_state_dict(
    checkpoint["model_state_dict"]
)

baseline_model.eval()

print("Baseline model restored successfully!")
print("Checkpoint epoch:", checkpoint["epoch"])
print("Validation Dice:", checkpoint["validation_dice"])
print("Device:", device)
print("Model parameters:", sum(
    p.numel() for p in baseline_model.parameters()
))


In [ ]:

import os

print("PENGWIN directory exists:",
      os.path.exists("/kaggle/working/PENGWIN"))

print("\nDirectory contents:")

for root, dirs, files in os.walk("/kaggle/working/PENGWIN"):
    level = root.replace(
        "/kaggle/working/PENGWIN", ""
    ).count(os.sep)

    if level > 2:
        dirs[:] = []
        continue

    print("\nFolder:", root)

    for filename in files[:10]:
        print("  ", filename)


In [ ]:

import os
import zipfile

DOWNLOAD_DIR = "/kaggle/working/PENGWIN/downloads"

zip_files = [
    "ct_part1.zip",
    "ct_part2.zip",
    "labels.zip"
]

for filename in zip_files:
    path = os.path.join(DOWNLOAD_DIR, filename)

    print("\nFile:", filename)

    if not os.path.exists(path):
        print("File missing!")
        continue

    print("Size:", round(os.path.getsize(path) / (1024**3), 3), "GB")

    with zipfile.ZipFile(path, "r") as z:
        names = z.namelist()

        print("Number of entries:", len(names))
        print("First 5 entries:")

        for name in names[:5]:
            print(" ", name)


In [ ]:

import os
import zipfile
import shutil
import SimpleITK as sitk

BASE = "/kaggle/working/PENGWIN"
DOWNLOAD_DIR = os.path.join(BASE, "downloads")
TEMP_DIR = os.path.join(BASE, "case_loader_test")

os.makedirs(TEMP_DIR, exist_ok=True)

def load_case(case_id):
    case_id = str(case_id).zfill(3)

    ct_zip = (
        "ct_part1.zip" if int(case_id) <= 50
        else "ct_part2.zip"
    )

    ct_zip_path = os.path.join(DOWNLOAD_DIR, ct_zip)
    label_zip_path = os.path.join(DOWNLOAD_DIR, "labels.zip")

    ct_path = os.path.join(TEMP_DIR, "ct.mha")
    label_path = os.path.join(TEMP_DIR, "label.mha")

    # Extract CT volume
    with zipfile.ZipFile(ct_zip_path, "r") as z:
        with z.open(f"{case_id}.mha") as source:
            with open(ct_path, "wb") as target:
                shutil.copyfileobj(source, target)

    # Extract corresponding segmentation label
    with zipfile.ZipFile(label_zip_path, "r") as z:
        with z.open(f"{case_id}.mha") as source:
            with open(label_path, "wb") as target:
                shutil.copyfileobj(source, target)

    ct_image = sitk.ReadImage(ct_path)
    label_image = sitk.ReadImage(label_path)

    return ct_image, label_image


print("Dataset loading function restored!")


In [ ]:
ct_image, label_image = load_case("001")

print("CT size:", ct_image.GetSize())
print("Label size:", label_image.GetSize())
print("CT spacing:", ct_image.GetSpacing())
print("Label spacing:", label_image.GetSpacing())

In [ ]:

import SimpleITK as sitk
import numpy as np

def resample_image(image, target_spacing=(1.0, 1.0, 1.0),
                   is_label=False):

    original_spacing = image.GetSpacing()
    original_size = image.GetSize()

    new_size = [
        int(round(
            original_size[i] *
            original_spacing[i] /
            target_spacing[i]
        ))
        for i in range(3)
    ]

    interpolator = (
        sitk.sitkNearestNeighbor
        if is_label
        else sitk.sitkLinear
    )

    resampled_image = sitk.Resample(
        image,
        new_size,
        sitk.Transform(),
        interpolator,
        image.GetOrigin(),
        target_spacing,
        image.GetDirection(),
        0,
        image.GetPixelID()
    )

    return resampled_image


print("Resampling function restored successfully!")


In [ ]:

ct_resampled = resample_image(
    ct_image,
    target_spacing=(1.0, 1.0, 1.0),
    is_label=False
)

label_resampled = resample_image(
    label_image,
    target_spacing=(1.0, 1.0, 1.0),
    is_label=True
)

print("Original CT size:", ct_image.GetSize())
print("Resampled CT size:", ct_resampled.GetSize())

print("\nOriginal spacing:", ct_image.GetSpacing())
print("Resampled spacing:", ct_resampled.GetSpacing())

print("\nResampled label size:", label_resampled.GetSize())
print("Label spacing:", label_resampled.GetSpacing())


In [ ]:

def preprocess_case(case_id, target_spacing=(1.0, 1.0, 1.0)):

    ct_image, label_image = load_case(case_id)

    # Verify original geometry
    assert ct_image.GetSize() == label_image.GetSize(), \
        "CT and label sizes do not match!"

    assert np.allclose(
        ct_image.GetSpacing(),
        label_image.GetSpacing()
    ), "CT and label spacing do not match!"

    assert np.allclose(
        ct_image.GetOrigin(),
        label_image.GetOrigin()
    ), "CT and label origins do not match!"

    # Resample CT and label
    ct_resampled = resample_image(
        ct_image, target_spacing, is_label=False
    )

    label_resampled = resample_image(
        label_image, target_spacing, is_label=True
    )

    # Convert to NumPy arrays (Z, Y, X)
    ct_array = sitk.GetArrayFromImage(ct_resampled)
    label_array = sitk.GetArrayFromImage(label_resampled)

    # CT clipping and normalization
    ct_array = np.clip(ct_array, -200, 2000)
    ct_array = (ct_array + 200) / 2200.0

    label_array = label_array.astype(np.int16)

    return ct_array.astype(np.float32), label_array


print("Preprocessing function restored successfully!")


In [ ]:

ct_volume, label_volume = preprocess_case("001")

print("CT shape:", ct_volume.shape)
print("Label shape:", label_volume.shape)

print("\nCT minimum:", ct_volume.min())
print("CT maximum:", ct_volume.max())

print("\nUnique label classes:",
      np.unique(label_volume))

print("\nForeground voxels:",
      int(np.sum(label_volume > 0)))

print("\nPreprocessing test completed!")


In [ ]:

import os
import json

SPLIT_PATH = "/kaggle/working/PENGWIN/dataset_split.json"

with open(SPLIT_PATH, "r") as f:
    dataset_split = json.load(f)

train_ids = dataset_split["train_ids"]
val_ids = dataset_split["val_ids"]
test_ids = dataset_split["test_ids"]

print("Dataset split restored successfully!")

print("\nTraining cases:", len(train_ids))
print("Validation cases:", len(val_ids))
print("Testing cases:", len(test_ids))

print("\nFirst 5 training cases:", train_ids[:5])
print("Validation cases:", val_ids)
print("First 5 testing cases:", test_ids[:5])

assert len(train_ids) == 70
assert len(val_ids) == 15
assert len(test_ids) == 15

assert not set(train_ids) & set(val_ids)
assert not set(train_ids) & set(test_ids)
assert not set(val_ids) & set(test_ids)

print("\nAll split checks passed!")


In [ ]:

import json

SPLIT_PATH = "/kaggle/working/PENGWIN/dataset_split.json"

with open(SPLIT_PATH, "r") as f:
    dataset_split = json.load(f)

print("JSON data type:", type(dataset_split))

print("\nAvailable keys:")
if isinstance(dataset_split, dict):
    print(list(dataset_split.keys()))

print("\nComplete JSON content:")
print(json.dumps(dataset_split, indent=4)[:5000])


In [ ]:

import json

SPLIT_PATH = "/kaggle/working/PENGWIN/dataset_split.json"

with open(SPLIT_PATH, "r") as f:
    dataset_split = json.load(f)

train_ids = dataset_split["train"]
val_ids = dataset_split["validation"]
test_ids = dataset_split["test"]

print("Dataset split restored successfully!")

print("\nTraining cases:", len(train_ids))
print("Validation cases:", len(val_ids))
print("Testing cases:", len(test_ids))

print("\nFirst 5 training cases:", train_ids[:5])
print("\nValidation cases:", val_ids)
print("\nFirst 5 testing cases:", test_ids[:5])

assert len(train_ids) == 70
assert len(val_ids) == 15
assert len(test_ids) == 15

assert not set(train_ids) & set(val_ids)
assert not set(train_ids) & set(test_ids)
assert not set(val_ids) & set(test_ids)

print("\nAll dataset split checks passed!")


In [ ]:

import os
import json

FIXED_PATCH_PATH = (
    "/kaggle/working/PENGWIN/fixed_validation_patches.json"
)

with open(FIXED_PATCH_PATH, "r") as f:
    fixed_patches = json.load(f)

print("Fixed validation patches restored!")

print("Number of validation patches:", len(fixed_patches))

print("\nFirst validation patch:")
first_case = list(fixed_patches.keys())[0]
print("Case:", first_case)
print("Coordinates:", fixed_patches[first_case])

assert len(fixed_patches) == 15

print("\nValidation patch restoration successful!")


In [ ]:

import torch
import numpy as np

baseline_model.eval()
baseline_dice_scores = []

with torch.no_grad():

    for case_id, patch_info in fixed_patches.items():

        ct, label = preprocess_case(case_id)

        z0 = patch_info["z"]
        y0 = patch_info["y"]
        x0 = patch_info["x"]
        patch_size = patch_info["patch_size"]

        ct_patch = ct[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        label_patch = label[
            z0:z0+patch_size,
            y0:y0+patch_size,
            x0:x0+patch_size
        ]

        ct_tensor = (
            torch.from_numpy(ct_patch.astype(np.float32))
            .unsqueeze(0)
            .unsqueeze(0)
            .to(device)
        )

        logits = baseline_model(ct_tensor)

        prediction_patch = (
            torch.argmax(logits, dim=1)
            .squeeze(0)
            .cpu()
            .numpy()
        )

        true_mask = label_patch > 0
        pred_mask = prediction_patch > 0

        intersection = np.logical_and(
            true_mask, pred_mask
        ).sum()

        denominator = true_mask.sum() + pred_mask.sum()

        dice = (
            2 * intersection / denominator
            if denominator > 0 else 1.0
        )

        baseline_dice_scores.append(float(dice))

        print(f"Case {case_id}: Dice = {dice:.4f}")

        del ct_tensor, logits, ct, label

recovered_baseline_dice = float(
    np.mean(baseline_dice_scores)
)

print("\nRecovered baseline mean validation Dice:",
      round(recovered_baseline_dice, 4))


In [ ]:

from monai.inferers import sliding_window_inference
import torch
import numpy as np

# Load complete validation case
ct, label = preprocess_case("001")

ct_tensor = (
    torch.from_numpy(ct.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(device)
)

baseline_model.eval()

# Full-volume prediction
with torch.no_grad():
    logits = sliding_window_inference(
        ct_tensor,
        roi_size=(96, 96, 96),
        sw_batch_size=1,
        predictor=baseline_model,
        overlap=0.25,
        mode="gaussian"
    )

prediction = (
    torch.argmax(logits, dim=1)
    .squeeze(0)
    .cpu()
    .numpy()
)

# Calculate foreground Dice
true_mask = label > 0
pred_mask = prediction > 0

intersection = np.logical_and(
    true_mask, pred_mask
).sum()

denominator = true_mask.sum() + pred_mask.sum()

full_volume_dice = (
    2 * intersection / denominator
    if denominator > 0 else 1.0
)

print("Case: 001")
print("CT shape:", ct.shape)
print("Ground-truth foreground voxels:", int(true_mask.sum()))
print("Predicted foreground voxels:", int(pred_mask.sum()))
print("Recovered baseline full-volume Dice:",
      round(float(full_volume_dice), 4))

del ct_tensor, logits, ct, label


In [ ]:

import numpy as np

# Restore ground-truth labels for case 001
_, label_image = load_case("001")

label_resampled = resample_image(
    label_image,
    target_spacing=(1.0, 1.0, 1.0),
    is_label=True
)

label = sitk.GetArrayFromImage(
    label_resampled
).astype(np.int16)

print("PREDICTED CLASS DISTRIBUTION")
print("=" * 50)

classes, counts = np.unique(
    prediction, return_counts=True
)

for class_id, count in zip(classes, counts):
    print(
        f"Class {class_id:2d}: "
        f"{count:,} voxels "
        f"({count / prediction.size * 100:.3f}%)"
    )

print("\nGROUND-TRUTH CLASS DISTRIBUTION")
print("=" * 50)

classes, counts = np.unique(
    label, return_counts=True
)

for class_id, count in zip(classes, counts):
    print(
        f"Class {class_id:2d}: "
        f"{count:,} voxels "
        f"({count / label.size * 100:.3f}%)"
    )


In [ ]:

import numpy as np

print("BASELINE CLASS-WISE EVALUATION")
print("=" * 65)

for class_id in [1, 11, 12, 21]:

    true_class = label == class_id
    pred_class = prediction == class_id

    tp = np.logical_and(true_class, pred_class).sum()
    fp = np.logical_and(~true_class, pred_class).sum()
    fn = np.logical_and(true_class, ~pred_class).sum()

    dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    print(f"\nClass {class_id}")
    print("TP:", int(tp))
    print("FP:", int(fp))
    print("FN:", int(fn))
    print("Dice:", round(float(dice), 4))
    print("Precision:", round(float(precision), 4))
    print("Recall:", round(float(recall), 4))


In [ ]:
checkpoint = torch.load(
    "/kaggle/working/PENGWIN/checkpoints/baseline_best.pth",
    map_location="cpu",
    weights_only=False
)

print("Checkpoint details")
print("=" * 40)

for key, value in checkpoint.items():
    if key != "model_state_dict" and key != "optimizer_state_dict":
        print(key, ":", value)

print("\nModel output channels:", baseline_model.out_channels)
print("Number of classes:", 31)

In [ ]:
print("TRAINING CLASS DISTRIBUTION")
print("=" * 55)

for case_id in ["043", "042", "092", "010", "066"]:
    ct, label = preprocess_case(case_id)

    print(f"\nCase: {case_id}")

    for class_id in [1, 11, 12, 21]:
        count = np.sum(label == class_id)
        print(f"Class {class_id}: {count:,} voxels")

    del ct, label

In [ ]:
from collections import defaultdict

class_totals = defaultdict(int)
case_presence = defaultdict(int)

for case_id in train_ids:
    ct, label = preprocess_case(case_id)

    for class_id in [1, 11, 12, 21]:
        count = int(np.sum(label == class_id))

        class_totals[class_id] += count

        if count > 0:
            case_presence[class_id] += 1

    del ct, label

print("TRAINING DATASET CLASS DISTRIBUTION")
print("=" * 55)

for class_id in [1, 11, 12, 21]:
    print(f"\nClass {class_id}")
    print("Total voxels:", f"{class_totals[class_id]:,}")
    print("Cases containing class:", case_presence[class_id], "/ 70")

In [ ]:
import random

random.seed(42)

case_id = "043"
ct, label = preprocess_case(case_id)

print("CLASS 12 PATCH INVESTIGATION")
print("=" * 50)

for i in range(10):
    patch_ct, patch_label = sample_balanced_patch(
        ct, label, patch_size=96, max_attempts=20
    )

    class12_voxels = np.sum(patch_label == 12)
    total_voxels = patch_label.size

    print(
        f"Patch {i+1}: "
        f"Class 12 = {class12_voxels:,} "
        f"({class12_voxels/total_voxels*100:.3f}%)"
    )

del ct, label

In [ ]:
import inspect

print(inspect.getsource(sample_balanced_patch))

In [ ]:
return ct_patch, label_patch, class_id

In [ ]:
import random
random.seed(42)
np.random.seed(42)

case_id = "043"
ct, label = preprocess_case(case_id)

print("CLASS 12 PATCH INVESTIGATION")
print("=" * 50)

for i in range(10):
    patch_ct, patch_label, selected_class = sample_balanced_patch(
        ct, label, patch_size=96, max_attempts=20
    )

    class12_voxels = np.sum(patch_label == 12)
    total_voxels = patch_label.size

    print(f"\nPatch {i+1}")
    print("Selected class:", selected_class)
    print("Class 12 voxels:", f"{class12_voxels:,}")
    print("Class 12 percentage:",
          f"{class12_voxels / total_voxels * 100:.3f}%")

del ct, label

In [ ]:
from collections import Counter

np.random.seed(42)

case_id = "043"
ct, label = preprocess_case(case_id)

selected_counts = Counter()
class12_present = 0
class12_total = 0

for i in range(100):
    patch_ct, patch_label, selected_class = sample_balanced_patch(
        ct, label, patch_size=96, max_attempts=20
    )

    selected_counts[selected_class] += 1

    count = int(np.sum(patch_label == 12))
    class12_total += count

    if count > 0:
        class12_present += 1

print("100 PATCH SAMPLING ANALYSIS")
print("=" * 45)

print("Selected class frequencies:")
for cls, count in sorted(selected_counts.items()):
    print(f"Class {cls}: {count} patches")

print("\nClass 12 present in:", class12_present, "/ 100 patches")
print("Class 12 absent in:", 100 - class12_present, "/ 100 patches")
print("Total Class 12 voxels:", f"{class12_total:,}")

del ct, label

In [ ]:
print("VALIDATION CLASS 12 DISTRIBUTION")
print("=" * 50)

total_class12 = 0
cases_with_class12 = 0

for case_id in val_ids:
    ct, label = preprocess_case(case_id)

    count = int(np.sum(label == 12))
    total_class12 += count

    if count > 0:
        cases_with_class12 += 1

    print(f"Case {case_id}: {count:,} voxels")

    del ct, label

print("\nSUMMARY")
print("Cases containing Class 12:",
      cases_with_class12, "/", len(val_ids))
print("Total Class 12 voxels:",
      f"{total_class12:,}")

In [ ]:
import os
import numpy as np
import torch

RESULT_DIR = "/kaggle/working/PENGWIN/final_results"
os.makedirs(RESULT_DIR, exist_ok=True)

np.save(
    os.path.join(RESULT_DIR, "baseline_prediction_case001.npy"),
    prediction
)

np.save(
    os.path.join(RESULT_DIR, "ground_truth_case001.npy"),
    label
)

print("Baseline results saved successfully!")
print("Prediction shape:", prediction.shape)
print("Ground truth shape:", label.shape)
print("Output directory:", RESULT_DIR)

In [ ]:
# Reload case 001 ground-truth label
ct, label = preprocess_case("001")

# Save baseline prediction and ground truth
np.save(
    os.path.join(RESULT_DIR, "baseline_prediction_case001.npy"),
    prediction
)

np.save(
    os.path.join(RESULT_DIR, "ground_truth_case001.npy"),
    label
)

print("Both files saved successfully!")
print("Prediction shape:", prediction.shape)
print("Ground truth shape:", label.shape)
print("Saved directory:", RESULT_DIR)

del ct, label

In [ ]:
import importlib.util

libraries = ["skimage", "trimesh", "matplotlib"]

for library in libraries:
    print(
        library,
        "Installed" if importlib.util.find_spec(library) else "Not installed"
    )

In [ ]:
import numpy as np
import trimesh
from skimage.measure import marching_cubes

# Load saved prediction
prediction = np.load(
    "/kaggle/working/PENGWIN/final_results/baseline_prediction_case001.npy"
)

# Select Class 11
bone_mask = (prediction == 11).astype(np.uint8)

print("Bone mask shape:", bone_mask.shape)
print("Bone voxels:", np.sum(bone_mask))

# Generate 3D surface
vertices, faces, normals, values = marching_cubes(
    bone_mask,
    level=0.5,
    spacing=(1.0, 1.0, 1.0)
)

# Create mesh
mesh = trimesh.Trimesh(
    vertices=vertices,
    faces=faces,
    vertex_normals=normals,
    process=True
)

print("\n3D MESH GENERATED")
print("Vertices:", len(mesh.vertices))
print("Faces:", len(mesh.faces))
print("Watertight:", mesh.is_watertight)
print("Volume:", mesh.volume)

In [ ]:
import trimesh
import numpy as np

# Split mesh into connected components
components = mesh.split(only_watertight=False)

print("Total connected components:", len(components))

# Sort components by number of faces
components = sorted(
    components,
    key=lambda component: len(component.faces),
    reverse=True
)

print("\nLargest components:")

for i, component in enumerate(components[:10]):
    print(
        f"Component {i+1}: "
        f"{len(component.vertices):,} vertices, "
        f"{len(component.faces):,} faces"
    )

# Keep largest component for initial inspection
clean_mesh = components[0].copy()

# Fix face winding and normals
trimesh.repair.fix_winding(clean_mesh)
trimesh.repair.fix_normals(clean_mesh)

print("\nCLEANED MESH")
print("Vertices:", len(clean_mesh.vertices))
print("Faces:", len(clean_mesh.faces))
print("Watertight:", clean_mesh.is_watertight)
print("Volume:", clean_mesh.volume)

In [ ]:
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(12, 9))
ax = fig.add_subplot(111, projection="3d")

ax.plot_trisurf(
    clean_mesh.vertices[:, 0],
    clean_mesh.vertices[:, 1],
    clean_mesh.vertices[:, 2],
    triangles=clean_mesh.faces,
    linewidth=0,
    alpha=0.9
)

ax.set_title("3D Reconstruction - Class 11")
ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")

plt.tight_layout()
plt.show()

In [ ]:
import os
import numpy as np
import trimesh

# Copy the cleaned mesh
export_mesh = clean_mesh.copy()

# Convert Z,Y,X coordinates into X,Y,Z
export_mesh.vertices = export_mesh.vertices[:, [2, 1, 0]]

# Correct normals and face orientation
trimesh.repair.fix_winding(export_mesh)
trimesh.repair.fix_normals(export_mesh)

# Export directory
STL_DIR = "/kaggle/working/PENGWIN/final_results/STL"
os.makedirs(STL_DIR, exist_ok=True)

# Export
stl_path = os.path.join(STL_DIR, "case001_class11_largest.stl")
export_mesh.export(stl_path)

print("STL EXPORT SUCCESSFUL")
print("File:", stl_path)
print("Vertices:", len(export_mesh.vertices))
print("Faces:", len(export_mesh.faces))
print("Watertight:", export_mesh.is_watertight)
print("Volume:", export_mesh.volume)
print("File size (MB):", round(os.path.getsize(stl_path)/1024**2, 2))

In [ ]:
import numpy as np
import trimesh
import os
from skimage.measure import marching_cubes

RESULT_DIR = "/kaggle/working/PENGWIN/final_results"
STL_DIR = os.path.join(RESULT_DIR, "STL")

# Load ground-truth segmentation
ground_truth = np.load(
    os.path.join(RESULT_DIR, "ground_truth_case001.npy")
)

# Extract Class 11
gt_mask = (ground_truth == 11).astype(np.uint8)

print("Ground-truth Class 11 voxels:", np.sum(gt_mask))

# Marching Cubes
gt_vertices, gt_faces, gt_normals, _ = marching_cubes(
    gt_mask,
    level=0.5,
    spacing=(1.0, 1.0, 1.0)
)

# Create mesh
gt_mesh = trimesh.Trimesh(
    vertices=gt_vertices[:, [2, 1, 0]],
    faces=gt_faces,
    process=True
)

# Repair orientation
trimesh.repair.fix_winding(gt_mesh)
trimesh.repair.fix_normals(gt_mesh)

# Export
gt_path = os.path.join(
    STL_DIR, "case001_class11_ground_truth.stl"
)

gt_mesh.export(gt_path)

print("\nGROUND-TRUTH STL EXPORTED")
print("Vertices:", len(gt_mesh.vertices))
print("Faces:", len(gt_mesh.faces))
print("Watertight:", gt_mesh.is_watertight)
print("Volume:", gt_mesh.volume)
print("File size MB:", round(os.path.getsize(gt_path)/1024**2, 2))

del ground_truth, gt_mask

In [ ]:
import trimesh
import numpy as np

# Split predicted and ground-truth meshes
pred_components = mesh.split(only_watertight=False)
gt_components = gt_mesh.split(only_watertight=False)

pred_components = sorted(
    pred_components,
    key=lambda m: len(m.faces),
    reverse=True
)

gt_components = sorted(
    gt_components,
    key=lambda m: len(m.faces),
    reverse=True
)

print("CONNECTED COMPONENT COMPARISON")
print("=" * 50)

print("\nPredicted mesh")
print("Total components:", len(pred_components))

print("\nGround-truth mesh")
print("Total components:", len(gt_components))

print("\nLargest 5 predicted components:")
for i, component in enumerate(pred_components[:5]):
    print(i+1, "Faces:", len(component.faces))

print("\nLargest 5 ground-truth components:")
for i, component in enumerate(gt_components[:5]):
    print(i+1, "Faces:", len(component.faces))

In [ ]:
import numpy as np
from scipy import ndimage

# Load prediction
prediction = np.load(
    "/kaggle/working/PENGWIN/final_results/baseline_prediction_case001.npy"
)

# Extract Class 11
bone_mask = (prediction == 11)

# Find 3D connected components
structure = np.ones((3, 3, 3), dtype=np.uint8)

labeled, num_components = ndimage.label(
    bone_mask,
    structure=structure
)

component_sizes = np.bincount(labeled.ravel())[1:]
component_sizes = np.sort(component_sizes)[::-1]

print("3D COMPONENT SIZE ANALYSIS")
print("=" * 50)
print("Total components:", num_components)

print("\nLargest 10 components (voxels):")
print(component_sizes[:10])

print("\nComponents above different size thresholds:")

for threshold in [100, 500, 1000, 5000, 10000]:
    count = np.sum(component_sizes >= threshold)
    voxels = component_sizes[component_sizes >= threshold].sum()

    print(
        f">= {threshold:5d} voxels: "
        f"{count:5d} components, "
        f"{voxels:,} voxels"
    )

In [ ]:
import numpy as np
from scipy import ndimage

# Load ground truth
ground_truth = np.load(
    "/kaggle/working/PENGWIN/final_results/ground_truth_case001.npy"
)

# Original Class 11 prediction
pred_mask = (prediction == 11)

# Ground-truth Class 11
gt_mask = (ground_truth == 11)

# Find connected components using 26-connectivity
structure = np.ones((3, 3, 3), dtype=np.uint8)

labeled, num = ndimage.label(
    pred_mask,
    structure=structure
)

sizes = np.bincount(labeled.ravel())

# Candidate: retain components >= 5000 voxels
keep = np.where(sizes >= 5000)[0]
keep = keep[keep != 0]

filtered_mask = np.isin(labeled, keep)

def evaluate_mask(pred, gt):
    tp = np.logical_and(pred, gt).sum()
    fp = np.logical_and(pred, ~gt).sum()
    fn = np.logical_and(~pred, gt).sum()

    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    return dice, precision, recall

original = evaluate_mask(pred_mask, gt_mask)
filtered = evaluate_mask(filtered_mask, gt_mask)

print("CLASS 11 FILTERING COMPARISON")
print("=" * 50)

print("\nOriginal prediction")
print("Voxels:", pred_mask.sum())
print("Dice, Precision, Recall:", original)

print("\nFiltered prediction (5000 voxels)")
print("Components retained:", len(keep))
print("Voxels:", filtered_mask.sum())
print("Dice, Precision, Recall:", filtered)

del ground_truth, gt_mask

In [ ]:
import os
import numpy as np

RESULT_DIR = "/kaggle/working/PENGWIN/final_results"

# Save filtered Class 11 mask
filtered_path = os.path.join(
    RESULT_DIR,
    "case001_class11_filtered.npy"
)

np.save(filtered_path, filtered_mask.astype(np.uint8))

print("Filtered segmentation saved!")
print("Path:", filtered_path)
print("Remaining voxels:", int(filtered_mask.sum()))

In [ ]:
import numpy as np
import trimesh
import os
from skimage.measure import marching_cubes

# Load filtered segmentation
filtered_mask = np.load(
    "/kaggle/working/PENGWIN/final_results/case001_class11_filtered.npy"
)

# Generate surface using Marching Cubes
vertices, faces, normals, values = marching_cubes(
    filtered_mask,
    level=0.5,
    spacing=(1.0, 1.0, 1.0)
)

# Convert Z,Y,X to X,Y,Z
vertices = vertices[:, [2, 1, 0]]

# Create mesh
filtered_mesh = trimesh.Trimesh(
    vertices=vertices,
    faces=faces,
    process=True
)

# Repair orientation
trimesh.repair.fix_winding(filtered_mesh)
trimesh.repair.fix_normals(filtered_mesh)

# Export STL
stl_path = (
    "/kaggle/working/PENGWIN/final_results/STL/"
    "case001_class11_filtered.stl"
)

filtered_mesh.export(stl_path)

print("FILTERED STL GENERATED")
print("Vertices:", len(filtered_mesh.vertices))
print("Faces:", len(filtered_mesh.faces))
print("Watertight:", filtered_mesh.is_watertight)
print("Volume:", filtered_mesh.volume)
print("File size MB:", round(os.path.getsize(stl_path)/1024**2, 2))

In [ ]:
import numpy as np
import trimesh
from scipy import ndimage

filtered_mask = np.load(
    "/kaggle/working/PENGWIN/final_results/case001_class11_filtered.npy"
)

# Check voxel components using 26-connectivity
structure = np.ones((3, 3, 3), dtype=np.uint8)

labels, count = ndimage.label(
    filtered_mask,
    structure=structure
)

print("FILTERED MASK")
print("Shape:", filtered_mask.shape)
print("Foreground voxels:", int(filtered_mask.sum()))
print("26-connected components:", count)

# Inspect mesh components
parts = filtered_mesh.split(only_watertight=False)

print("\nFILTERED MESH")
print("Mesh components:", len(parts))
print("Watertight components:",
      sum(part.is_watertight for part in parts))
print("Non-watertight components:",
      sum(not part.is_watertight for part in parts))
print("Euler number:", filtered_mesh.euler_number)
print("Is winding consistent:", filtered_mesh.is_winding_consistent)

In [ ]:
import numpy as np
from scipy import ndimage

filtered_mask = np.load(
    "/kaggle/working/PENGWIN/final_results/case001_class11_filtered.npy"
)

connectivity_structures = {
    "6-connectivity": ndimage.generate_binary_structure(3, 1),
    "18-connectivity": ndimage.generate_binary_structure(3, 2),
    "26-connectivity": ndimage.generate_binary_structure(3, 3)
}

print("VOXEL CONNECTIVITY COMPARISON")
print("=" * 45)

for name, structure in connectivity_structures.items():

    labeled, count = ndimage.label(
        filtered_mask,
        structure=structure
    )

    sizes = np.bincount(labeled.ravel())[1:]

    print(f"\n{name}")
    print("Components:", count)
    print("Largest component:", sizes.max())
    print("Components >= 5000 voxels:",
          np.sum(sizes >= 5000))

In [ ]:
import os
import numpy as np
import trimesh
from scipy import ndimage
from skimage.measure import marching_cubes

mask = np.load(
    "/kaggle/working/PENGWIN/final_results/case001_class11_filtered.npy"
).astype(bool)

# Use 6-connectivity
structure = ndimage.generate_binary_structure(3, 1)
labels, count = ndimage.label(mask, structure=structure)

sizes = np.bincount(labels.ravel())
component_ids = np.where(sizes >= 5000)[0]
component_ids = component_ids[component_ids != 0]

MESH_DIR = "/kaggle/working/PENGWIN/final_results/individual_meshes"
os.makedirs(MESH_DIR, exist_ok=True)

results = []

for index, component_id in enumerate(component_ids, start=1):

    # Get component bounding box
    slices = ndimage.find_objects(
        labels == component_id
    )[0]

    z0, y0, x0 = [s.start for s in slices]
    z1, y1, x1 = [s.stop for s in slices]

    # Extract and pad component
    component = (
        labels[z0:z1, y0:y1, x0:x1] == component_id
    )

    component = np.pad(component, 1)

    vertices, faces, _, _ = marching_cubes(
        component.astype(np.uint8),
        level=0.5,
        spacing=(1.0, 1.0, 1.0)
    )

    # Convert Z,Y,X to X,Y,Z and restore position
    vertices = vertices[:, [2, 1, 0]]
    vertices += np.array([x0, y0, z0]) - 1

    fragment_mesh = trimesh.Trimesh(
        vertices=vertices,
        faces=faces,
        process=True
    )

    trimesh.repair.fix_winding(fragment_mesh)
    trimesh.repair.fix_normals(fragment_mesh)

    path = os.path.join(
        MESH_DIR,
        f"class11_fragment_{index:02d}.stl"
    )

    fragment_mesh.export(path)

    results.append(fragment_mesh)

    print(
        f"Fragment {index}: "
        f"Voxels={sizes[component_id]:,}, "
        f"Watertight={fragment_mesh.is_watertight}"
    )

print("\nTOTAL FRAGMENTS:", len(results))
print("WATERTIGHT FRAGMENTS:",
      sum(m.is_watertight for m in results))
print("OUTPUT DIRECTORY:", MESH_DIR)

In [ ]:
import trimesh
import os

REPAIR_DIR = (
    "/kaggle/working/PENGWIN/final_results/"
    "repaired_meshes"
)

os.makedirs(REPAIR_DIR, exist_ok=True)

repair_results = []

for i, original_mesh in enumerate(results, start=1):

    repaired = original_mesh.copy()

    # Fix face orientation and normals
    trimesh.repair.fix_winding(repaired)
    trimesh.repair.fix_normals(repaired)

    # Attempt to close holes
    trimesh.repair.fill_holes(repaired)

    # Check mesh quality
    is_watertight = repaired.is_watertight

    # Export repaired fragment
    output_path = os.path.join(
        REPAIR_DIR,
        f"repaired_fragment_{i:02d}.stl"
    )

    repaired.export(output_path)

    repair_results.append(repaired)

    print(
        f"Fragment {i:02d}: "
        f"Watertight = {is_watertight}, "
        f"Faces = {len(repaired.faces):,}"
    )

print("\nREPAIR SUMMARY")
print("Total fragments:", len(repair_results))
print(
    "Watertight after repair:",
    sum(m.is_watertight for m in repair_results)
)
print("Output:", REPAIR_DIR)

In [ ]:
# Load case 001 at original preprocessed resolution
ct, label = preprocess_case("001")

print("ADAPTIVE COARSE-TO-FINE INITIALIZATION")
print("=" * 50)

print("CT shape:", ct.shape)
print("Ground truth shape:", label.shape)
print("Model:", type(baseline_model).__name__)
print("Device:", device)
print("Model training parameters:", 
      sum(p.numel() for p in baseline_model.parameters()))

print("CT minimum:", float(ct.min()))
print("CT maximum:", float(ct.max()))

print("\nInitialization successful!")

In [ ]:
import torch
import torch.nn.functional as F
from monai.inferers import sliding_window_inference
import numpy as np

baseline_model.eval()

# Convert CT to GPU tensor
ct_tensor = torch.from_numpy(ct).float()
ct_tensor = ct_tensor.unsqueeze(0).unsqueeze(0).to(device)

# Downsample by a factor of 2
coarse_ct = F.interpolate(
    ct_tensor,
    scale_factor=0.5,
    mode="trilinear",
    align_corners=False
)

print("Original CT:", tuple(ct_tensor.shape))
print("Coarse CT:", tuple(coarse_ct.shape))

# Coarse segmentation
with torch.no_grad():
    coarse_logits = sliding_window_inference(
        inputs=coarse_ct,
        roi_size=(96, 96, 96),
        sw_batch_size=1,
        predictor=baseline_model,
        overlap=0.5
    )

    coarse_prediction = torch.argmax(
        coarse_logits, dim=1
    )

# Upsample segmentation to original resolution
coarse_prediction_full = F.interpolate(
    coarse_prediction.unsqueeze(1).float(),
    size=ct.shape,
    mode="nearest"
).squeeze(1)

coarse_prediction_full = (
    coarse_prediction_full[0].cpu().numpy().astype(np.uint8)
)

print("\nCOARSE SEGMENTATION COMPLETED")
print("Prediction shape:", coarse_prediction_full.shape)
print("Predicted classes:", np.unique(coarse_prediction_full))
print("Foreground voxels:", int(np.sum(coarse_prediction_full > 0)))

In [ ]:
import numpy as np

# Load original baseline prediction
baseline_prediction = np.load(
    "/kaggle/working/PENGWIN/final_results/"
    "baseline_prediction_case001.npy"
)

# Ground truth is already available as label

def calculate_metrics(pred, gt, class_id):
    p = pred == class_id
    g = gt == class_id

    tp = np.logical_and(p, g).sum()
    fp = np.logical_and(p, ~g).sum()
    fn = np.logical_and(~p, g).sum()

    dice = 2 * tp / (2 * tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    return dice, precision, recall

print("BASELINE vs COARSE SEGMENTATION")
print("=" * 75)

for class_id in [1, 11, 12, 21]:

    baseline_metrics = calculate_metrics(
        baseline_prediction, label, class_id
    )

    coarse_metrics = calculate_metrics(
        coarse_prediction_full, label, class_id
    )

    print(f"\nClass {class_id}")
    print(
        f"Baseline: Dice={baseline_metrics[0]:.4f}, "
        f"Precision={baseline_metrics[1]:.4f}, "
        f"Recall={baseline_metrics[2]:.4f}"
    )

    print(
        f"Coarse:   Dice={coarse_metrics[0]:.4f}, "
        f"Precision={coarse_metrics[1]:.4f}, "
        f"Recall={coarse_metrics[2]:.4f}"
    )

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

# Convert coarse CT to model input
baseline_model.eval()

with torch.no_grad():
    probabilities = torch.softmax(coarse_logits, dim=1)

    # Predictive entropy
    entropy = -torch.sum(
        probabilities * torch.log(probabilities + 1e-8),
        dim=1
    )

    # Normalize entropy to [0, 1]
    normalized_entropy = entropy / np.log(31)

    # Upsample uncertainty to original CT dimensions
    uncertainty = F.interpolate(
        normalized_entropy.unsqueeze(1),
        size=ct.shape,
        mode="trilinear",
        align_corners=False
    )[0, 0].cpu().numpy()

print("UNCERTAINTY ANALYSIS")
print("=" * 50)

print("Uncertainty shape:", uncertainty.shape)
print("Minimum:", float(uncertainty.min()))
print("Maximum:", float(uncertainty.max()))
print("Mean:", float(uncertainty.mean()))

for threshold in [0.2, 0.3, 0.4, 0.5]:
    selected = uncertainty > threshold

    print(
        f"Threshold {threshold}: "
        f"{selected.sum():,} voxels "
        f"({selected.mean()*100:.2f}%)"
    )

In [ ]:
import numpy as np
from scipy import ndimage

# Select uncertain voxels
uncertainty_threshold = 0.3

uncertain_mask = uncertainty > uncertainty_threshold

# Find 3D connected uncertain regions
structure = np.ones((3, 3, 3), dtype=np.uint8)

region_labels, num_regions = ndimage.label(
    uncertain_mask,
    structure=structure
)

# Extract bounding boxes
region_slices = ndimage.find_objects(region_labels)

roi_boxes = []

for region_id, slices in enumerate(region_slices, start=1):

    if slices is None:
        continue

    z0, y0, x0 = [s.start for s in slices]
    z1, y1, x1 = [s.stop for s in slices]

    roi_boxes.append({
        "region_id": region_id,
        "z": (z0, z1),
        "y": (y0, y1),
        "x": (x0, x1),
        "size": int(np.sum(region_labels[
            z0:z1, y0:y1, x0:x1
        ] == region_id))
    })

# Sort by uncertain voxel count
roi_boxes.sort(
    key=lambda box: box["size"],
    reverse=True
)

print("ADAPTIVE ROI SELECTION")
print("=" * 50)
print("Total uncertain regions:", num_regions)
print("Valid ROI boxes:", len(roi_boxes))

print("\nTop 10 regions:")

for roi in roi_boxes[:10]:
    print(
        f"Region {roi['region_id']}: "
        f"Z={roi['z']}, "
        f"Y={roi['y']}, "
        f"X={roi['x']}, "
        f"Uncertain voxels={roi['size']:,}"
    )

In [ ]:
import numpy as np

# Fixed patch configuration
PATCH_SIZE = 96
STRIDE = 64
MIN_UNCERTAIN_VOXELS = 5000

D, H, W = uncertainty.shape

roi_boxes = []

# Generate patch starting coordinates
z_starts = list(range(0, max(D - PATCH_SIZE + 1, 1), STRIDE))
y_starts = list(range(0, max(H - PATCH_SIZE + 1, 1), STRIDE))
x_starts = list(range(0, max(W - PATCH_SIZE + 1, 1), STRIDE))

# Ensure final patches cover volume boundaries
z_starts.append(max(D - PATCH_SIZE, 0))
y_starts.append(max(H - PATCH_SIZE, 0))
x_starts.append(max(W - PATCH_SIZE, 0))

z_starts = sorted(set(z_starts))
y_starts = sorted(set(y_starts))
x_starts = sorted(set(x_starts))

for z in z_starts:
    for y in y_starts:
        for x in x_starts:

            z1 = min(z + PATCH_SIZE, D)
            y1 = min(y + PATCH_SIZE, H)
            x1 = min(x + PATCH_SIZE, W)

            uncertain_count = int(
                np.sum(uncertainty[z:z1, y:y1, x:x1] > 0.3)
            )

            if uncertain_count >= MIN_UNCERTAIN_VOXELS:
                roi_boxes.append({
                    "z": (z, z1),
                    "y": (y, y1),
                    "x": (x, x1),
                    "uncertain_voxels": uncertain_count
                })

# Sort by uncertainty
roi_boxes.sort(
    key=lambda item: item["uncertain_voxels"],
    reverse=True
)

print("ADAPTIVE FIXED-SIZE ROI RESULTS")
print("=" * 50)

print("Total candidate patches:",
      len(z_starts) * len(y_starts) * len(x_starts))

print("Selected ROIs:", len(roi_boxes))

print("\nTop 10 selected ROIs:")

for i, roi in enumerate(roi_boxes[:10], start=1):
    print(
        f"ROI {i}: "
        f"Z={roi['z']}, "
        f"Y={roi['y']}, "
        f"X={roi['x']}, "
        f"Uncertain={roi['uncertain_voxels']:,}"
    )

In [ ]:
import torch
import numpy as np
from monai.inferers import sliding_window_inference

# Start with the coarse prediction
refined_prediction = coarse_prediction_full.copy()

# Track voxels already refined
refined_mask = np.zeros(ct.shape, dtype=bool)

# Select top 10 ROIs
selected_rois = roi_boxes[:10]

baseline_model.eval()

print("Starting high-resolution ROI refinement...")
print("Selected ROIs:", len(selected_rois))

for i, roi in enumerate(selected_rois, start=1):

    z0, z1 = roi["z"]
    y0, y1 = roi["y"]
    x0, x1 = roi["x"]

    # Extract original-resolution CT patch
    ct_patch = ct[z0:z1, y0:y1, x0:x1]

    input_tensor = torch.from_numpy(
        ct_patch.copy()
    ).float().unsqueeze(0).unsqueeze(0).to(device)

    # Full-resolution inference
    with torch.no_grad():
        logits = sliding_window_inference(
            inputs=input_tensor,
            roi_size=(96, 96, 96),
            sw_batch_size=1,
            predictor=baseline_model,
            overlap=0.25
        )

        patch_prediction = torch.argmax(
            logits, dim=1
        )[0].cpu().numpy().astype(np.uint8)

    # Update only voxels not already refined
    current_mask = refined_mask[z0:z1, y0:y1, x0:x1]
    update_mask = ~current_mask

    current_prediction = refined_prediction[z0:z1, y0:y1, x0:x1]

    current_prediction[update_mask] = patch_prediction[update_mask]

    current_mask[update_mask] = True

    print(
        f"ROI {i}/10 completed | "
        f"Uncertain voxels: {roi['uncertain_voxels']:,}"
    )

print("\nRefinement completed!")
print("Refined voxels:", int(refined_mask.sum()))
print("Total volume voxels:", refined_mask.size)
print("Refined percentage:", round(refined_mask.mean() * 100, 2), "%")
print("Predicted classes:", np.unique(refined_prediction))

In [ ]:
import numpy as np

def calculate_metrics(pred, gt, class_id):
    pred_mask = (pred == class_id)
    gt_mask = (gt == class_id)

    tp = np.logical_and(pred_mask, gt_mask).sum()
    fp = np.logical_and(pred_mask, ~gt_mask).sum()
    fn = np.logical_and(~pred_mask, gt_mask).sum()

    dice = (2 * tp) / (2 * tp + fp + fn + 1e-8)
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    return dice, precision, recall


print("=" * 75)
print("COARSE vs ADAPTIVE REFINED SEGMENTATION")
print("=" * 75)

print(f"{'Class':<10}{'Metric':<15}{'Coarse':<15}{'Refined':<15}")
print("-" * 75)

for class_id in [1, 11, 12, 21]:

    coarse_metrics = calculate_metrics(
        coarse_prediction_full, label, class_id
    )

    refined_metrics = calculate_metrics(
        refined_prediction, label, class_id
    )

    for metric_name, coarse_value, refined_value in zip(
        ["Dice", "Precision", "Recall"],
        coarse_metrics,
        refined_metrics
    ):
        print(
            f"{class_id:<10}{metric_name:<15}"
            f"{coarse_value:<15.4f}{refined_value:<15.4f}"
        )

    print("-" * 75)

# Save adaptive prediction
np.save(
    RESULT_DIR + "/case001_adaptive_prediction.npy",
    refined_prediction
)

print("\nAdaptive prediction saved successfully!")

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np

baseline_model.eval()

with torch.no_grad():
    coarse_probabilities = torch.softmax(
        coarse_logits, dim=1
    )

    # Upsample probabilities to original CT dimensions
    coarse_probabilities_full = F.interpolate(
        coarse_probabilities,
        size=ct.shape,
        mode="trilinear",
        align_corners=False
    )[0].cpu().numpy()

print("Coarse probability shape:", coarse_probabilities_full.shape)
print("Expected shape:", (31,) + ct.shape)
print("Probability range:",
      coarse_probabilities_full.min(),
      coarse_probabilities_full.max())

print("Probability preparation completed!")

In [ ]:
# Store high-resolution probability maps for selected ROIs
refined_roi_probabilities = []

baseline_model.eval()

print("Generating high-resolution ROI probabilities...")

for i, roi in enumerate(selected_rois, start=1):

    z0, z1 = roi["z"]
    y0, y1 = roi["y"]
    x0, x1 = roi["x"]

    ct_patch = ct[z0:z1, y0:y1, x0:x1]

    input_tensor = torch.from_numpy(
        ct_patch.copy()
    ).float().unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = sliding_window_inference(
            inputs=input_tensor,
            roi_size=(96, 96, 96),
            sw_batch_size=1,
            predictor=baseline_model,
            overlap=0.25
        )

        probabilities = torch.softmax(logits, dim=1)[0]

        probabilities = probabilities.cpu().numpy()

    refined_roi_probabilities.append({
        "z": roi["z"],
        "y": roi["y"],
        "x": roi["x"],
        "probabilities": probabilities
    })

    print(f"ROI {i}/10 completed: {probabilities.shape}")

print("\nAll high-resolution ROI probabilities generated!")

In [ ]:
# Probability fusion
COARSE_WEIGHT = 0.70
REFINED_WEIGHT = 0.30

# Start with the coarse segmentation
fused_prediction = coarse_prediction_full.copy()

# Prevent duplicate updates in overlapping ROIs
fusion_mask = np.zeros(ct.shape, dtype=bool)

print("Starting probability fusion...")
print("=" * 50)

for i, roi_data in enumerate(refined_roi_probabilities, start=1):

    z0, z1 = roi_data["z"]
    y0, y1 = roi_data["y"]
    x0, x1 = roi_data["x"]

    # Coarse probabilities for this ROI
    coarse_roi = coarse_probabilities_full[
        :, z0:z1, y0:y1, x0:x1
    ]

    # High-resolution probabilities
    refined_roi = roi_data["probabilities"]

    # Weighted probability fusion
    blended_probabilities = (
        COARSE_WEIGHT * coarse_roi +
        REFINED_WEIGHT * refined_roi
    )

    # Predict the most probable class
    roi_prediction = np.argmax(
        blended_probabilities, axis=0
    ).astype(np.uint8)

    # Update only voxels not covered by earlier ROIs
    current_mask = fusion_mask[z0:z1, y0:y1, x0:x1]
    update_mask = ~current_mask

    current_prediction = fused_prediction[
        z0:z1, y0:y1, x0:x1
    ]

    current_prediction[update_mask] = roi_prediction[update_mask]
    current_mask[update_mask] = True

    print(f"ROI {i}/10 fused successfully")

print("\nProbability fusion completed!")
print("Fused voxels:", int(fusion_mask.sum()))
print("Fused percentage:", round(fusion_mask.mean() * 100, 2), "%")
print("Final predicted classes:", np.unique(fused_prediction))

# Save fused prediction
np.save(
    RESULT_DIR + "/case001_fused_prediction.npy",
    fused_prediction
)

print("Fused prediction saved!")

In [ ]:
import pandas as pd

results = []

for class_id in [1, 11, 12, 21]:

    for method_name, prediction in [
        ("Coarse", coarse_prediction_full),
        ("Direct Refinement", refined_prediction),
        ("Probability Fusion", fused_prediction)
    ]:

        dice, precision, recall = calculate_metrics(
            prediction, label, class_id
        )

        results.append({
            "Class": class_id,
            "Method": method_name,
            "Dice": round(dice, 4),
            "Precision": round(precision, 4),
            "Recall": round(recall, 4)
        })

results_df = pd.DataFrame(results)

print("SEGMENTATION COMPARISON")
display(results_df)

print("\nMEAN DICE BY METHOD")
display(
    results_df.groupby("Method")["Dice"]
    .mean()
    .sort_values(ascending=False)
    .to_frame("Mean Dice")
)

In [ ]:
# Load original CT image geometry for case 001
original_ct_image, original_label_image = load_case("001")

# Resample to 1 mm spacing
resampled_ct_image = resample_image(
    original_ct_image,
    target_spacing=(1, 1, 1),
    is_label=False
)

print("Original spacing:",
      original_ct_image.GetSpacing())

print("Resampled spacing:",
      resampled_ct_image.GetSpacing())

print("Origin:",
      resampled_ct_image.GetOrigin())

print("Direction:",
      resampled_ct_image.GetDirection())

print("Image size:",
      resampled_ct_image.GetSize())

In [ ]:
import numpy as np
from scipy import ndimage

# Group predicted labels by anatomical region
bone_regions = {
    "Sacrum": (1, 10),
    "Left_Hipbone": (11, 20),
    "Right_Hipbone": (21, 30)
}

print("3D BONE REGION ANALYSIS")
print("=" * 55)

for bone_name, (low, high) in bone_regions.items():

    bone_mask = (
        (fused_prediction >= low) &
        (fused_prediction <= high)
    )

    voxel_count = int(bone_mask.sum())

    # Count connected components using 26-connectivity
    structure = np.ones((3, 3, 3), dtype=np.uint8)

    component_labels, num_components = ndimage.label(
        bone_mask,
        structure=structure
    )

    component_sizes = np.bincount(
        component_labels.ravel()
    )[1:]

    largest_components = sorted(
        component_sizes,
        reverse=True
    )[:5]

    print(f"\n{bone_name}")
    print("Total voxels:", f"{voxel_count:,}")
    print("Connected components:", num_components)
    print("Largest component sizes:", largest_components)

print("\nBone region analysis completed!")

In [ ]:
import numpy as np
from scipy import ndimage

bone_regions = {
    "Sacrum": (1, 10),
    "Left_Hipbone": (11, 20),
    "Right_Hipbone": (21, 30)
}

thresholds = [100, 500, 1000, 2000, 5000]

structure = np.ones((3, 3, 3), dtype=np.uint8)

print("BONE FRAGMENT FILTERING ANALYSIS")
print("=" * 70)

for bone_name, (low, high) in bone_regions.items():

    bone_mask = (
        (fused_prediction >= low) &
        (fused_prediction <= high)
    )

    component_labels, num_components = ndimage.label(
        bone_mask,
        structure=structure
    )

    component_sizes = np.bincount(
        component_labels.ravel()
    )[1:]

    print(f"\n{bone_name}")
    print("Original components:", num_components)

    for threshold in thresholds:
        retained = component_sizes[component_sizes >= threshold]

        print(
            f"Threshold >= {threshold:5d}: "
            f"{len(retained):4d} components | "
            f"{retained.sum():,} voxels retained"
        )

print("\nAnalysis completed!")

In [ ]:
import numpy as np
from scipy import ndimage
import os

MIN_COMPONENT_SIZE = 1000

bone_regions = {
    "Sacrum": (1, 10),
    "Left_Hipbone": (11, 20),
    "Right_Hipbone": (21, 30)
}

FILTERED_DIR = RESULT_DIR + "/filtered_bones"
os.makedirs(FILTERED_DIR, exist_ok=True)

structure = np.ones((3, 3, 3), dtype=np.uint8)

filtered_bones = {}

print("FILTERING BONE FRAGMENTS")
print("=" * 55)

for bone_name, (low, high) in bone_regions.items():

    bone_mask = (
        (fused_prediction >= low) &
        (fused_prediction <= high)
    )

    component_labels, num_components = ndimage.label(
        bone_mask,
        structure=structure
    )

    component_sizes = np.bincount(
        component_labels.ravel()
    )

    # Keep components with at least 1000 voxels
    valid_components = np.where(
        component_sizes >= MIN_COMPONENT_SIZE
    )[0]

    valid_components = valid_components[
        valid_components != 0
    ]

    filtered_mask = np.isin(
        component_labels,
        valid_components
    )

    filtered_bones[bone_name] = filtered_mask

    np.save(
        os.path.join(FILTERED_DIR, bone_name + ".npy"),
        filtered_mask.astype(np.uint8)
    )

    print(f"\n{bone_name}")
    print("Retained fragments:", len(valid_components))
    print("Retained voxels:", int(filtered_mask.sum()))
    print("Saved successfully!")

print("\nAll filtered bone masks saved!")

In [ ]:
import numpy as np
import trimesh
import os
from skimage.measure import marching_cubes

STL_DIR = RESULT_DIR + "/Adaptive_STL"
os.makedirs(STL_DIR, exist_ok=True)

# Physical geometry from original CT
spacing = resampled_ct_image.GetSpacing()
origin = np.array(resampled_ct_image.GetOrigin())

bone_meshes = {}

print("GENERATING 3D BONE MESHES")
print("=" * 55)

for bone_name, bone_mask in filtered_bones.items():

    print(f"\nProcessing {bone_name}...")

    # Marching Cubes returns coordinates in Z,Y,X order
    vertices, faces, normals, values = marching_cubes(
        bone_mask.astype(np.uint8),
        level=0.5,
        spacing=(spacing[2], spacing[1], spacing[0])
    )

    # Convert Z,Y,X to X,Y,Z
    vertices_xyz = vertices[:, [2, 1, 0]]

    # Apply physical origin
    vertices_xyz += origin

    mesh = trimesh.Trimesh(
        vertices=vertices_xyz,
        faces=faces,
        process=True
    )

    # Store mesh
    bone_meshes[bone_name] = mesh

    # Export STL
    output_path = os.path.join(
        STL_DIR,
        bone_name + ".stl"
    )

    mesh.export(output_path)

    print("Vertices:", len(mesh.vertices))
    print("Faces:", len(mesh.faces))
    print("Watertight:", mesh.is_watertight)
    print("Volume:", mesh.volume)
    print("STL saved:", output_path)

print("\n3D mesh generation completed!")

In [ ]:
print("3D MESH QUALITY DIAGNOSTICS")
print("=" * 60)

for bone_name, mesh in bone_meshes.items():

    print(f"\n{bone_name}")
    print("-" * 40)

    print("Watertight:", mesh.is_watertight)
    print("Winding consistent:", mesh.is_winding_consistent)
    print("Euler number:", mesh.euler_number)

    print("Boundary edges:", len(mesh.edges_boundary)
          if hasattr(mesh, "edges_boundary") else "Not available")

    # Separate disconnected mesh components
    components = mesh.split(only_watertight=False)

    print("Connected mesh components:", len(components))

    watertight_count = sum(
        component.is_watertight
        for component in components
    )

    print("Watertight components:", watertight_count)
    print("Non-watertight components:",
          len(components) - watertight_count)

    print("Total surface area:", round(mesh.area, 2))
    print("Bounding box:", mesh.bounds)

print("\nDiagnostics completed!")

In [ ]:
import trimesh
import os

REPAIR_DIR = STL_DIR + "/repair_candidates"
os.makedirs(REPAIR_DIR, exist_ok=True)

defective_components = {}

print("IDENTIFYING NON-WATERTIGHT COMPONENTS")
print("=" * 60)

for bone_name, mesh in bone_meshes.items():

    components = mesh.split(only_watertight=False)

    defective = []

    for index, component in enumerate(components):

        if not component.is_watertight:

            defective.append(component)

            component_path = os.path.join(
                REPAIR_DIR,
                f"{bone_name}_defect_{index+1}.stl"
            )

            component.export(component_path)

    defective_components[bone_name] = defective

    print(f"\n{bone_name}")
    print("Defective components:", len(defective))

    for i, component in enumerate(defective, start=1):
        print(
            f"Defect {i}: "
            f"Vertices={len(component.vertices)}, "
            f"Faces={len(component.faces)}, "
            f"Euler={component.euler_number}, "
            f"Area={component.area:.2f}"
        )

print("\nDefective components extracted!")

In [ ]:
import numpy as np

print("BOUNDARY EDGE ANALYSIS")
print("=" * 65)

for bone_name, components in defective_components.items():

    print(f"\n{bone_name}")
    print("-" * 50)

    for i, component in enumerate(components, start=1):

        # Count how many faces share each unique edge
        edges = np.sort(component.edges, axis=1)

        unique_edges, edge_counts = np.unique(
            edges,
            axis=0,
            return_counts=True
        )

        boundary_count = int(np.sum(edge_counts == 1))
        non_manifold_count = int(np.sum(edge_counts > 2))

        print(
            f"Defect {i}: "
            f"Boundary edges={boundary_count:,}, "
            f"Non-manifold edges={non_manifold_count:,}, "
            f"Vertices={len(component.vertices):,}, "
            f"Faces={len(component.faces):,}"
        )

print("\nBoundary analysis completed!")

In [ ]:
import trimesh
import copy
import os

REPAIRED_DIR = STL_DIR + "/repaired_candidates"
os.makedirs(REPAIRED_DIR, exist_ok=True)

repaired_components = {}

print("TARGETED MESH REPAIR")
print("=" * 60)

for bone_name, components in defective_components.items():

    repaired_components[bone_name] = []

    print(f"\n{bone_name}")
    print("-" * 45)

    for i, original in enumerate(components, start=1):

        # Work on a copy
        repaired = original.copy()

        before_watertight = repaired.is_watertight

        # Attempt to fill suitable holes
        trimesh.repair.fill_holes(repaired)

        # Correct face normals
        trimesh.repair.fix_normals(repaired)

        after_watertight = repaired.is_watertight

        repaired_components[bone_name].append(repaired)

        print(
            f"Defect {i}: "
            f"Before={before_watertight}, "
            f"After={after_watertight}, "
            f"Boundary edges before="
            f"{len(original.edges) - len(original.edges_unique)}"
        )

        output_path = os.path.join(
            REPAIRED_DIR,
            f"{bone_name}_repaired_{i}.stl"
        )

        repaired.export(output_path)

print("\nRepair experiment completed!")
print("Repaired candidate files:", REPAIRED_DIR)

In [ ]:
import numpy as np
import trimesh
import os
from scipy import ndimage
from skimage.measure import marching_cubes

FRAGMENT_DIR = STL_DIR + "/individual_fragments"
os.makedirs(FRAGMENT_DIR, exist_ok=True)

spacing = resampled_ct_image.GetSpacing()
origin = np.array(resampled_ct_image.GetOrigin())

structure = np.ones((3, 3, 3), dtype=np.uint8)

individual_meshes = {}
fragment_summary = []

print("INDIVIDUAL 3D FRAGMENT RECONSTRUCTION")
print("=" * 60)

for bone_name, bone_mask in filtered_bones.items():

    component_labels, num_components = ndimage.label(
        bone_mask,
        structure=structure
    )

    component_slices = ndimage.find_objects(component_labels)

    individual_meshes[bone_name] = []

    for component_id, slices in enumerate(
        component_slices, start=1
    ):

        if slices is None:
            continue

        # Extract individual component
        component_mask = (
            component_labels[slices] == component_id
        )

        voxel_count = int(component_mask.sum())

        if voxel_count < MIN_COMPONENT_SIZE:
            continue

        # Add padding around the component
        padded_mask = np.pad(
            component_mask.astype(np.uint8),
            pad_width=1,
            mode="constant",
            constant_values=0
        )

        # Marching Cubes
        vertices, faces, normals, values = marching_cubes(
            padded_mask,
            level=0.5,
            spacing=(spacing[2], spacing[1], spacing[0])
        )

        # Remove padding offset and add original crop offset
        crop_offset = np.array([
            slices[0].start,
            slices[1].start,
            slices[2].start
        ])

        vertices_zyx = vertices / np.array([
            spacing[2], spacing[1], spacing[0]
        ])

        vertices_zyx -= 1
        vertices_zyx += crop_offset

        # Convert Z,Y,X to X,Y,Z and apply physical spacing
        vertices_xyz = vertices_zyx[:, [2, 1, 0]]
        vertices_xyz *= np.array(spacing)
        vertices_xyz += origin

        mesh = trimesh.Trimesh(
            vertices=vertices_xyz,
            faces=faces,
            process=True
        )

        filename = (
            f"{bone_name}_fragment_{component_id:02d}.stl"
        )

        output_path = os.path.join(
            FRAGMENT_DIR, filename
        )

        mesh.export(output_path)

        individual_meshes[bone_name].append(mesh)

        fragment_summary.append({
            "bone": bone_name,
            "fragment": component_id,
            "voxels": voxel_count,
            "vertices": len(mesh.vertices),
            "faces": len(mesh.faces),
            "watertight": mesh.is_watertight
        })

    print(f"\n{bone_name}")
    print("Fragments reconstructed:",
          len(individual_meshes[bone_name]))

print("\nRECONSTRUCTION SUMMARY")
print("=" * 60)

total = len(fragment_summary)
watertight = sum(
    item["watertight"] for item in fragment_summary
)

print("Total fragments:", total)
print("Watertight fragments:", watertight)
print("Non-watertight fragments:", total - watertight)
print("Output folder:", FRAGMENT_DIR)

In [ ]:
print("NON-WATERTIGHT FRAGMENT DIAGNOSTICS")
print("=" * 65)

for bone_name, meshes in individual_meshes.items():

    print(f"\n{bone_name}")
    print("-" * 50)

    for i, mesh in enumerate(meshes, start=1):

        if mesh.is_watertight:
            continue

        edges = np.sort(mesh.edges, axis=1)

        unique_edges, counts = np.unique(
            edges,
            axis=0,
            return_counts=True
        )

        boundary_edges = int(np.sum(counts == 1))
        non_manifold_edges = int(np.sum(counts > 2))

        print(
            f"Fragment {i}: "
            f"Vertices={len(mesh.vertices)}, "
            f"Faces={len(mesh.faces)}, "
            f"Boundary={boundary_edges}, "
            f"Non-manifold={non_manifold_edges}, "
            f"Euler={mesh.euler_number}"
        )

print("\nDiagnosis completed!")

In [ ]:
import numpy as np
import trimesh
import os
from scipy import ndimage
from skimage.measure import marching_cubes

SIX_CONNECTED = ndimage.generate_binary_structure(3, 1)

SIX_DIR = STL_DIR + "/six_connected_fragments"
os.makedirs(SIX_DIR, exist_ok=True)

six_connected_meshes = {}

print("6-CONNECTIVITY FRAGMENT RECONSTRUCTION")
print("=" * 60)

for bone_name, bone_mask in filtered_bones.items():

    component_labels, num_components = ndimage.label(
        bone_mask,
        structure=SIX_CONNECTED
    )

    component_slices = ndimage.find_objects(component_labels)

    six_connected_meshes[bone_name] = []

    print(f"\n{bone_name}")
    print("6-connected components:", num_components)

    for component_id, slices in enumerate(
        component_slices, start=1
    ):

        if slices is None:
            continue

        component_mask = (
            component_labels[slices] == component_id
        )

        if not component_mask.any():
            continue

        padded_mask = np.pad(
            component_mask.astype(np.uint8),
            1,
            mode="constant"
        )

        vertices, faces, normals, values = marching_cubes(
            padded_mask,
            level=0.5,
            spacing=(spacing[2], spacing[1], spacing[0])
        )

        crop_offset = np.array([
            slices[0].start,
            slices[1].start,
            slices[2].start
        ])

        vertices_zyx = vertices / np.array([
            spacing[2], spacing[1], spacing[0]
        ])

        vertices_zyx = vertices_zyx - 1 + crop_offset

        vertices_xyz = vertices_zyx[:, [2, 1, 0]]
        vertices_xyz *= np.array(spacing)
        vertices_xyz += origin

        mesh = trimesh.Trimesh(
            vertices=vertices_xyz,
            faces=faces,
            process=True
        )

        six_connected_meshes[bone_name].append(mesh)

        filename = (
            f"{bone_name}_6conn_{component_id:03d}.stl"
        )

        mesh.export(os.path.join(SIX_DIR, filename))

    print(
        "Watertight meshes:",
        sum(m.is_watertight for m in six_connected_meshes[bone_name])
    )

    print(
        "Non-watertight meshes:",
        sum(not m.is_watertight for m in six_connected_meshes[bone_name])
    )

print("\n6-connectivity reconstruction completed!")
print("Saved to:", SIX_DIR)

In [ ]:
import numpy as np
from scipy import ndimage

structure_6 = ndimage.generate_binary_structure(3, 1)

thresholds = [100, 500, 1000, 5000]

print("6-CONNECTIVITY COMPONENT SIZE ANALYSIS")
print("=" * 65)

for bone_name, bone_mask in filtered_bones.items():

    labels_6, count_6 = ndimage.label(
        bone_mask,
        structure=structure_6
    )

    sizes = np.bincount(labels_6.ravel())[1:]

    print(f"\n{bone_name}")
    print("Total components:", count_6)

    for threshold in thresholds:

        retained = sizes[sizes >= threshold]

        print(
            f"Size >= {threshold:5d}: "
            f"{len(retained):4d} components | "
            f"{retained.sum():,} voxels"
        )

    print("Largest 10 components:",
          sorted(sizes, reverse=True)[:10])

print("\nComponent size analysis completed!")

In [ ]:
import numpy as np
import trimesh
import os
from scipy import ndimage
from skimage.measure import marching_cubes

FINAL_FRAGMENT_DIR = STL_DIR + "/final_fragments"
os.makedirs(FINAL_FRAGMENT_DIR, exist_ok=True)

MIN_SIZE = 1000

structure_6 = ndimage.generate_binary_structure(3, 1)

final_meshes = {}
final_summary = []

print("FINAL SELECTED FRAGMENT RECONSTRUCTION")
print("=" * 65)

for bone_name, bone_mask in filtered_bones.items():

    labels_6, num_components = ndimage.label(
        bone_mask,
        structure=structure_6
    )

    sizes = np.bincount(labels_6.ravel())

    valid_ids = np.where(sizes >= MIN_SIZE)[0]
    valid_ids = valid_ids[valid_ids != 0]

    final_meshes[bone_name] = []

    for fragment_number, component_id in enumerate(
        valid_ids, start=1
    ):

        slices = ndimage.find_objects(labels_6 == component_id)[0]

        component_mask = (
            labels_6[slices] == component_id
        )

        padded_mask = np.pad(
            component_mask.astype(np.uint8),
            1,
            mode="constant"
        )

        vertices, faces, normals, values = marching_cubes(
            padded_mask,
            level=0.5,
            spacing=(spacing[2], spacing[1], spacing[0])
        )

        crop_offset = np.array([
            slices[0].start,
            slices[1].start,
            slices[2].start
        ])

        vertices_zyx = vertices / np.array([
            spacing[2], spacing[1], spacing[0]
        ])

        vertices_zyx = vertices_zyx - 1 + crop_offset

        vertices_xyz = vertices_zyx[:, [2, 1, 0]]
        vertices_xyz *= np.array(spacing)
        vertices_xyz += origin

        mesh = trimesh.Trimesh(
            vertices=vertices_xyz,
            faces=faces,
            process=True
        )

        filename = (
            f"{bone_name}_fragment_{fragment_number:02d}.stl"
        )

        output_path = os.path.join(
            FINAL_FRAGMENT_DIR, filename
        )

        mesh.export(output_path)

        final_meshes[bone_name].append(mesh)

        final_summary.append({
            "bone": bone_name,
            "fragment": fragment_number,
            "voxels": int(sizes[component_id]),
            "vertices": len(mesh.vertices),
            "faces": len(mesh.faces),
            "watertight": mesh.is_watertight
        })

    print(f"\n{bone_name}")
    print("Selected fragments:", len(valid_ids))
    print("Watertight meshes:", sum(
        m.is_watertight for m in final_meshes[bone_name]
    ))

print("\nFINAL RECONSTRUCTION SUMMARY")
print("=" * 65)

print("Total fragments:", len(final_summary))
print("Watertight fragments:", sum(
    x["watertight"] for x in final_summary
))
print("Non-watertight fragments:", sum(
    not x["watertight"] for x in final_summary
))

print("Saved directory:", FINAL_FRAGMENT_DIR)

In [ ]:
import os
import trimesh
import zipfile
import pandas as pd

FINAL_DIR = STL_DIR + "/FINAL_PROJECT_OUTPUT"
os.makedirs(FINAL_DIR, exist_ok=True)

print("CREATING COMBINED BONE MODELS")
print("=" * 55)

combined_meshes = {}

for bone_name, meshes in final_meshes.items():

    combined = trimesh.util.concatenate(meshes)

    combined_meshes[bone_name] = combined

    output_path = os.path.join(
        FINAL_DIR,
        bone_name + "_Combined.stl"
    )

    combined.export(output_path)

    print(f"\n{bone_name}")
    print("Fragments:", len(meshes))
    print("Vertices:", len(combined.vertices))
    print("Faces:", len(combined.faces))
    print("Watertight:", combined.is_watertight)
    print("Saved:", output_path)


# Save fragment summary
summary_df = pd.DataFrame(final_summary)

summary_path = os.path.join(
    FINAL_DIR,
    "fragment_quality_summary.csv"
)

summary_df.to_csv(summary_path, index=False)


# Create ZIP package
zip_path = os.path.join(
    FINAL_DIR,
    "AI_3D_Bone_Reconstruction_Results.zip"
)

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as zipf:

    # Include individual STL files
    for root, dirs, files in os.walk(FINAL_FRAGMENT_DIR):
        for file in files:
            if file.endswith(".stl"):
                full_path = os.path.join(root, file)
                arcname = os.path.join(
                    "Individual_Fragments", file
                )
                zipf.write(full_path, arcname)

    # Include combined STL files
    for bone_name in combined_meshes:
        file_path = os.path.join(
            FINAL_DIR, bone_name + "_Combined.stl"
        )
        zipf.write(
            file_path,
            os.path.basename(file_path)
        )

    # Include CSV summary
    zipf.write(
        summary_path,
        "fragment_quality_summary.csv"
    )

print("\n" + "=" * 55)
print("FINAL PACKAGE CREATED")
print("ZIP:", zip_path)
print("Summary:", summary_path)
print("Total individual fragments:", len(final_summary))
print("Combined anatomical models:", len(combined_meshes))

In [ ]:
import plotly.graph_objects as go
import numpy as np

fig = go.Figure()

bone_colors = {
    "Sacrum": "royalblue",
    "Left_Hipbone": "orange",
    "Right_Hipbone": "seagreen"
}

for bone_name, mesh in combined_meshes.items():

    # Reduce faces for faster visualization only
    step = max(1, len(mesh.faces) // 25000)
    faces = mesh.faces[::step]

    fig.add_trace(go.Mesh3d(
        x=mesh.vertices[:, 0],
        y=mesh.vertices[:, 1],
        z=mesh.vertices[:, 2],
        i=faces[:, 0],
        j=faces[:, 1],
        k=faces[:, 2],
        name=bone_name,
        color=bone_colors[bone_name],
        opacity=0.85
    ))

fig.update_layout(
    title="AI-Based 3D Pelvic Bone Reconstruction",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data"
    ),
    width=1000,
    height=750
)

fig.show()

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

colors = {
    "Sacrum": "royalblue",
    "Left_Hipbone": "orange",
    "Right_Hipbone": "seagreen"
}

for bone_name, original_mesh in combined_meshes.items():

    print("Processing:", bone_name)

    try:
        mesh = original_mesh.simplify_quadric_decimation(
            face_count=30000
        )
    except Exception as e:
        print("Simplification unavailable:", e)
        mesh = original_mesh

    fig.add_trace(go.Mesh3d(
        x=mesh.vertices[:, 0],
        y=mesh.vertices[:, 1],
        z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0],
        j=mesh.faces[:, 1],
        k=mesh.faces[:, 2],
        color=colors[bone_name],
        name=bone_name,
        opacity=0.95,
        flatshading=False
    ))

fig.update_layout(
    title="Adaptive AI-Based 3D Pelvic Bone Reconstruction",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data"
    ),
    width=1100,
    height=850
)

fig.show()

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

PLOT_DIR = "/kaggle/working/PENGWIN/Paper_Plots"
os.makedirs(PLOT_DIR, exist_ok=True)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 11

print("Plot directory created:", PLOT_DIR)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# Store generated figures
figures = []

def save_plot(fig, filename):
    path = os.path.join(PLOT_DIR, filename)
    fig.savefig(path, dpi=300, bbox_inches="tight")
    figures.append(path)
    plt.show()
    plt.close(fig)
    print("Saved:", filename)


# =====================================================
# FIGURE 1: PATCH VALIDATION DICE
# =====================================================

experiments = [
    "Recovered baseline",
    "Balanced sampling",
    "Earlier baseline",
    "Class-aware fine-tuning"
]

dice_scores = [0.8892, 0.9104, 0.9042, 0.8995]

fig, ax = plt.subplots()

bars = ax.bar(experiments, dice_scores)

ax.set_title("Patch Validation Dice Comparison")
ax.set_ylabel("Dice Score")
ax.set_ylim(0, 1)
ax.tick_params(axis="x", rotation=15)
ax.grid(axis="y", alpha=0.3)

for bar, score in zip(bars, dice_scores):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        score + 0.01,
        f"{score:.4f}",
        ha="center"
    )

save_plot(fig, "Figure_1_Patch_Dice.png")


# =====================================================
# FIGURE 2: CLASS-WISE DICE COMPARISON
# =====================================================

classes = ["Class 1", "Class 11", "Class 12", "Class 21"]

coarse = [0.4692, 0.6040, 0.0004, 0.5270]
direct = [0.4643, 0.5367, 0.0000, 0.5608]
fusion = [0.5286, 0.6482, 0.0000, 0.5734]

x = np.arange(len(classes))
width = 0.25

fig, ax = plt.subplots()

ax.bar(x-width, coarse, width, label="Coarse")
ax.bar(x, direct, width, label="Direct Refinement")
ax.bar(x+width, fusion, width, label="Probability Fusion")

ax.set_title("Case 001 Full-Volume Class-wise Dice")
ax.set_ylabel("Dice Score")
ax.set_xticks(x)
ax.set_xticklabels(classes)
ax.set_ylim(0, 0.75)
ax.legend()
ax.grid(axis="y", alpha=0.3)

save_plot(fig, "Figure_2_Classwise_Dice.png")


# =====================================================
# FIGURE 3: MEAN DICE COMPARISON
# =====================================================

methods = [
    "Coarse",
    "Direct Refinement",
    "Probability Fusion"
]

mean_dice = [0.40015, 0.39045, 0.43755]

fig, ax = plt.subplots()

bars = ax.bar(methods, mean_dice)

ax.set_title("Case 001 Full-Volume Mean Dice")
ax.set_ylabel("Mean Dice")
ax.set_ylim(0, 0.5)
ax.grid(axis="y", alpha=0.3)

for bar, score in zip(bars, mean_dice):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        score + 0.008,
        f"{score:.5f}",
        ha="center"
    )

save_plot(fig, "Figure_3_Mean_Dice.png")


# =====================================================
# FIGURE 4: ADAPTIVE ROI COVERAGE
# =====================================================

roi_percentage = 11.74
remaining_percentage = 100 - roi_percentage

fig, ax = plt.subplots()

values = [roi_percentage, remaining_percentage]

bars = ax.bar(
    ["High-resolution ROI", "Remaining Volume"],
    values
)

ax.set_title("Adaptive ROI Refinement Coverage")
ax.set_ylabel("Volume Percentage")
ax.set_ylim(0, 110)
ax.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, values):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 1,
        f"{value:.2f}%",
        ha="center"
    )

save_plot(fig, "Figure_4_ROI_Coverage.png")


# =====================================================
# FIGURE 5: STL MESH QUALITY
# =====================================================

bones = ["Sacrum", "Left Hipbone", "Right Hipbone"]

watertight = [15, 5, 13]
non_watertight = [0, 2, 6]

fig, ax = plt.subplots()

ax.bar(bones, watertight, label="Watertight")
ax.bar(
    bones,
    non_watertight,
    bottom=watertight,
    label="Non-watertight"
)

ax.set_title("Individual STL Fragment Quality")
ax.set_ylabel("Number of Fragments")
ax.legend()
ax.grid(axis="y", alpha=0.3)

save_plot(fig, "Figure_5_STL_Quality.png")


# =====================================================
# FIGURE 6: TRAINING CLASS DISTRIBUTION
# =====================================================

class_names = ["Class 1", "Class 11", "Class 12", "Class 21"]

voxel_counts = [
    12092571,
    18614040,
    1139585,
    18679047
]

fig, ax = plt.subplots()

bars = ax.bar(class_names, voxel_counts)

ax.set_title("Training Dataset Class Distribution")
ax.set_ylabel("Number of Voxels")
ax.ticklabel_format(axis="y", style="plain")
ax.grid(axis="y", alpha=0.3)

for bar, value in zip(bars, voxel_counts):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        value + 200000,
        f"{value/1e6:.2f}M",
        ha="center"
    )

save_plot(fig, "Figure_6_Class_Distribution.png")


# =====================================================
# FIGURE 7: COMPONENT FILTERING
# =====================================================

thresholds = [100, 500, 1000, 2000, 5000]

sacrum = [488898, 482446, 475109, 470372, 448092]
left_hip = [573412, 568857, 567346, 565077, 557615]
right_hip = [477078, 471575, 468483, 460917, 440688]

fig, ax = plt.subplots()

ax.plot(thresholds, sacrum, marker="o", label="Sacrum")
ax.plot(thresholds, left_hip, marker="o", label="Left Hipbone")
ax.plot(thresholds, right_hip, marker="o", label="Right Hipbone")

ax.set_title("Bone Fragment Filtering Analysis")
ax.set_xlabel("Minimum Component Size (Voxels)")
ax.set_ylabel("Retained Voxels")
ax.legend()
ax.grid(alpha=0.3)

save_plot(fig, "Figure_7_Component_Filtering.png")


print("\nALL 7 RESEARCH PLOTS GENERATED SUCCESSFULLY!")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Find the axial slice with the most ground-truth bone
foreground_per_slice = np.sum(label > 0, axis=(1, 2))
z = int(np.argmax(foreground_per_slice))

print("Selected CT slice:", z)
print("Ground-truth foreground voxels:",
      foreground_per_slice[z])

# Prepare four visualizations
fig, axes = plt.subplots(1, 4, figsize=(22, 6))

# 1. Original CT
axes[0].imshow(ct[z], cmap="gray")
axes[0].set_title("Original CT Scan")

# 2. Ground truth
axes[1].imshow(ct[z], cmap="gray")
axes[1].imshow(
    np.ma.masked_where(label[z] == 0, label[z]),
    cmap="tab20",
    alpha=0.65,
    vmin=1,
    vmax=30
)
axes[1].set_title("Ground Truth Segmentation")

# 3. Coarse prediction
axes[2].imshow(ct[z], cmap="gray")
axes[2].imshow(
    np.ma.masked_where(
        coarse_prediction_full[z] == 0,
        coarse_prediction_full[z]
    ),
    cmap="tab20",
    alpha=0.65,
    vmin=1,
    vmax=30
)
axes[2].set_title("Coarse AI Prediction")

# 4. Adaptive prediction
axes[3].imshow(ct[z], cmap="gray")
axes[3].imshow(
    np.ma.masked_where(
        fused_prediction[z] == 0,
        fused_prediction[z]
    ),
    cmap="tab20",
    alpha=0.65,
    vmin=1,
    vmax=30
)
axes[3].set_title("Adaptive Fusion Prediction")

for ax in axes:
    ax.axis("off")

plt.suptitle(
    "AI-Based Pelvic Bone Segmentation: Case 001",
    fontsize=16
)

plt.tight_layout()
plt.savefig(
    PLOT_DIR + "/Figure_8_CT_Segmentation_Comparison.png",
    dpi=300,
    bbox_inches="tight"
)
plt.show()

In [ ]:
import numpy as np
import pandas as pd

results = []

for cls in range(1, 31):
    gt_mask = (label == cls)
    pred_mask = (fused_prediction == cls)

    gt_voxels = np.sum(gt_mask)
    pred_voxels = np.sum(pred_mask)

    intersection = np.sum(gt_mask & pred_mask)

    dice = (
        2 * intersection / (gt_voxels + pred_voxels)
        if (gt_voxels + pred_voxels) > 0
        else 0
    )

    results.append({
        "Fragment ID": cls,
        "Ground Truth Present": gt_voxels > 0,
        "AI Predicted": pred_voxels > 0,
        "GT Voxels": int(gt_voxels),
        "AI Voxels": int(pred_voxels),
        "Dice Score": round(dice, 4)
    })

fragment_df = pd.DataFrame(results)

print("FRAGMENT-WISE AI IDENTIFICATION RESULTS")
display(fragment_df)

print("\nGround truth fragments:",
      int(fragment_df["Ground Truth Present"].sum()))

print("AI predicted fragments:",
      int(fragment_df["AI Predicted"].sum()))

print("\nMissed fragments:")
display(fragment_df[
    fragment_df["Ground Truth Present"] &
    (~fragment_df["AI Predicted"])
])

In [ ]:
import os

BASE = "/kaggle/working/PENGWIN"

PLOT_DIR = os.path.join(BASE, "Paper_Plots")
RESULT_DIR = os.path.join(BASE, "final_results")

folders = {
    "01_CT_Segmentation": "CT and AI segmentation plots",
    "02_Model_Performance": "Dice, precision and recall",
    "03_Adaptive_Refinement": "Uncertainty and ROI plots",
    "04_3D_Reconstruction": "Complete pelvis and fragment plots",
    "05_Mesh_Quality": "STL and mesh quality plots",
    "06_Final_Summary": "Final research result plots"
}

os.makedirs(PLOT_DIR, exist_ok=True)

for folder, description in folders.items():
    path = os.path.join(PLOT_DIR, folder)
    os.makedirs(path, exist_ok=True)
    print(folder, "-> READY")

print("\n========== PROJECT RESULT CHECK ==========")

variables = [
    "ct",
    "label",
    "coarse_prediction_full",
    "fused_prediction",
    "bone_groups",
    "final_meshes",
    "final_summary",
    "metrics_df",
    "summary_df",
    "region_df",
    "fragment_df"
]

for name in variables:
    print(name, ":", "AVAILABLE" if name in globals() else "NOT IN MEMORY")

print("\nPlot folder:", PLOT_DIR)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

OUT = os.path.join(PLOT_DIR, "01_CT_Segmentation")
os.makedirs(OUT, exist_ok=True)

# Select the axial slice containing the most annotated bone voxels
slice_counts = np.count_nonzero(label, axis=(1, 2))
z = int(np.argmax(slice_counts))

ct_slice = ct[z]
gt_slice = label[z]
coarse_slice = coarse_prediction_full[z]
adaptive_slice = fused_prediction[z]

# Create distinct colors for fragment labels
colors = plt.cm.tab20(np.linspace(0, 1, 30))
cmap = ListedColormap(colors)

fig, axes = plt.subplots(1, 5, figsize=(25, 6))

# 1. Original CT
axes[0].imshow(ct_slice, cmap="gray")
axes[0].set_title("Original CT Scan")

# 2. Ground truth
axes[1].imshow(ct_slice, cmap="gray")
axes[1].imshow(
    np.ma.masked_where(gt_slice == 0, gt_slice),
    cmap=cmap, vmin=1, vmax=30, alpha=0.75
)
axes[1].set_title("Ground Truth")

# 3. Coarse prediction
axes[2].imshow(ct_slice, cmap="gray")
axes[2].imshow(
    np.ma.masked_where(coarse_slice == 0, coarse_slice),
    cmap=cmap, vmin=1, vmax=30, alpha=0.75
)
axes[2].set_title("Coarse AI Prediction")

# 4. Adaptive prediction
axes[3].imshow(ct_slice, cmap="gray")
axes[3].imshow(
    np.ma.masked_where(adaptive_slice == 0, adaptive_slice),
    cmap=cmap, vmin=1, vmax=30, alpha=0.75
)
axes[3].set_title("Adaptive Fusion Prediction")

# 5. Disagreement map
disagreement = (
    (gt_slice != adaptive_slice) &
    ((gt_slice != 0) | (adaptive_slice != 0))
)

axes[4].imshow(ct_slice, cmap="gray")
axes[4].imshow(
    np.ma.masked_where(~disagreement, disagreement),
    cmap=ListedColormap(["yellow"]),
    alpha=0.8
)
axes[4].set_title("GT vs AI Disagreement")

for ax in axes:
    ax.axis("off")

fig.suptitle(
    f"AI-Based Pelvic Bone Segmentation – Case 001 – Slice {z}",
    fontsize=16
)

plt.tight_layout()

save_path = os.path.join(
    OUT, "Figure_01_CT_Segmentation_Comparison.png"
)

plt.savefig(save_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure 1 saved:", save_path)
print("Selected axial slice:", z)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUT = os.path.join(PLOT_DIR, "02_Model_Performance")
os.makedirs(OUT, exist_ok=True)

results = []

for cls in range(1, 31):

    gt_mask = (label == cls)
    gt_count = np.count_nonzero(gt_mask)

    for model_name, pred in [
        ("Coarse", coarse_prediction_full),
        ("Adaptive Fusion", fused_prediction)
    ]:

        pred_mask = (pred == cls)
        pred_count = np.count_nonzero(pred_mask)

        intersection = np.count_nonzero(gt_mask & pred_mask)

        dice = (
            2 * intersection / (gt_count + pred_count)
            if gt_count + pred_count > 0 else 0
        )

        precision = (
            intersection / pred_count if pred_count > 0 else 0
        )

        recall = (
            intersection / gt_count if gt_count > 0 else 0
        )

        results.append({
            "Fragment_ID": cls,
            "Model": model_name,
            "GT_Present": gt_count > 0,
            "AI_Predicted": pred_count > 0,
            "Dice": dice,
            "Precision": precision,
            "Recall": recall
        })

metrics_df = pd.DataFrame(results)

# Save all fragment metrics
metrics_df.to_csv(
    os.path.join(OUT, "All_30_Fragment_Metrics.csv"),
    index=False
)

# Mean Dice for each model, using GT-present labels
mean_dice = (
    metrics_df[metrics_df["GT_Present"]]
    .groupby("Model")["Dice"].mean()
)

# Figure 2: Overall mean Dice
plt.figure(figsize=(7, 5))
mean_dice.reindex(["Coarse", "Adaptive Fusion"]).plot(
    kind="bar", rot=0
)
plt.ylabel("Mean Dice Score")
plt.ylim(0, 1)
plt.title("Coarse vs Adaptive Fusion – Case 001")
plt.tight_layout()
plt.savefig(
    os.path.join(OUT, "Figure_02_Mean_Dice_Comparison.png"),
    dpi=300
)
plt.show()

# Figure 3: Fragment-wise Dice
plt.figure(figsize=(14, 6))

for model in ["Coarse", "Adaptive Fusion"]:
    df = metrics_df[metrics_df["Model"] == model]
    plt.plot(
        df["Fragment_ID"], df["Dice"],
        marker="o", label=model
    )

plt.xticks(range(1, 31))
plt.ylim(0, 1)
plt.xlabel("Fragment ID")
plt.ylabel("Dice Score")
plt.title("Fragment-wise Dice Comparison")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(OUT, "Figure_03_Fragment_Dice.png"),
    dpi=300
)
plt.show()

# Figure 4: Precision and Recall
adaptive_df = metrics_df[
    metrics_df["Model"] == "Adaptive Fusion"
]

plt.figure(figsize=(14, 6))
plt.plot(
    adaptive_df["Fragment_ID"],
    adaptive_df["Precision"],
    marker="o", label="Precision"
)
plt.plot(
    adaptive_df["Fragment_ID"],
    adaptive_df["Recall"],
    marker="s", label="Recall"
)

plt.xticks(range(1, 31))
plt.ylim(0, 1)
plt.xlabel("Fragment ID")
plt.ylabel("Score")
plt.title("Adaptive Fusion – Fragment-wise Precision and Recall")
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig(
    os.path.join(OUT, "Figure_04_Precision_Recall.png"),
    dpi=300
)
plt.show()

# Figure 5: Improvement per fragment
pivot = metrics_df.pivot(
    index="Fragment_ID",
    columns="Model",
    values="Dice"
)

improvement = pivot["Adaptive Fusion"] - pivot["Coarse"]

plt.figure(figsize=(14, 6))
plt.bar(improvement.index, improvement.values)
plt.axhline(0, linestyle="--")
plt.xticks(range(1, 31))
plt.xlabel("Fragment ID")
plt.ylabel("Dice Improvement")
plt.title("Adaptive Fusion Dice Improvement per Fragment")
plt.tight_layout()
plt.savefig(
    os.path.join(OUT, "Figure_05_Fragment_Improvement.png"),
    dpi=300
)
plt.show()

print("========== CASE 001 SUMMARY ==========")
print(mean_dice)

print("\nAdaptive fusion better on:",
      int((improvement > 0).sum()), "fragments")

print("Adaptive fusion worse on:",
      int((improvement < 0).sum()), "fragments")

print("\nAll Stage 2 plots saved in:", OUT)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import ndimage as ndi

# Output folder
METRIC_DIR = os.path.join(
    "/kaggle/working/PENGWIN",
    "Paper_Plots",
    "02_Model_Performance"
)
os.makedirs(METRIC_DIR, exist_ok=True)

# Verify existing variables
assert "label" in globals()
assert "coarse_prediction_full" in globals()
assert "fused_prediction" in globals()

GT = label
PREDICTIONS = {
    "Coarse": coarse_prediction_full,
    "Adaptive Fusion": fused_prediction
}

# Resampled voxel spacing in Z, Y, X order
SPACING = (1.0, 1.0, 1.0)


def calculate_metrics(gt, pred, spacing=SPACING):

    gt = np.asarray(gt, dtype=bool)
    pred = np.asarray(pred, dtype=bool)

    intersection = np.count_nonzero(gt & pred)
    gt_count = np.count_nonzero(gt)
    pred_count = np.count_nonzero(pred)

    # Dice
    dice = (
        2 * intersection / (gt_count + pred_count)
        if gt_count + pred_count > 0 else np.nan
    )

    # IoU
    union = np.count_nonzero(gt | pred)
    iou = intersection / union if union > 0 else np.nan

    # Precision
    precision = (
        intersection / pred_count if pred_count > 0 else 0.0
    )

    # Recall
    recall = (
        intersection / gt_count if gt_count > 0 else 0.0
    )

    # Surface distance metrics
    if gt_count == 0 and pred_count == 0:
        hd95 = np.nan
        assd = np.nan

    elif gt_count == 0 or pred_count == 0:
        hd95 = np.inf
        assd = np.inf

    else:

        # Extract surface voxels
        structure = ndi.generate_binary_structure(3, 1)

        gt_surface = gt & ~ndi.binary_erosion(
            gt, structure=structure, border_value=0
        )

        pred_surface = pred & ~ndi.binary_erosion(
            pred, structure=structure, border_value=0
        )

        # Distance from each surface to the opposite surface
        distance_to_gt = ndi.distance_transform_edt(
            ~gt_surface, sampling=spacing
        )

        distance_to_pred = ndi.distance_transform_edt(
            ~pred_surface, sampling=spacing
        )

        distances_pred_to_gt = distance_to_gt[pred_surface]
        distances_gt_to_pred = distance_to_pred[gt_surface]

        all_distances = np.concatenate([
            distances_pred_to_gt,
            distances_gt_to_pred
        ])

        hd95 = np.percentile(all_distances, 95)

        assd = (
            distances_pred_to_gt.mean()
            + distances_gt_to_pred.mean()
        ) / 2

    return {
        "Dice": dice,
        "IoU": iou,
        "Precision": precision,
        "Recall": recall,
        "HD95_mm": hd95,
        "ASSD_mm": assd,
        "GT_Voxels": gt_count,
        "Predicted_Voxels": pred_count
    }


# Calculate metrics for all 30 fragment labels
results = []

for cls in range(1, 31):

    gt_mask = GT == cls

    # Skip classes absent from ground truth
    if not np.any(gt_mask):
        continue

    for model_name, prediction in PREDICTIONS.items():

        pred_mask = prediction == cls

        metric = calculate_metrics(gt_mask, pred_mask)

        results.append({
            "Fragment_ID": cls,
            "Model": model_name,
            **metric
        })

metrics_df = pd.DataFrame(results)

# Save detailed metrics
csv_path = os.path.join(
    METRIC_DIR,
    "Case001_Complete_Segmentation_Metrics.csv"
)

metrics_df.to_csv(csv_path, index=False)

# Display all fragment-level results
print("========== CASE 001: FRAGMENT-WISE METRICS ==========")
display(metrics_df.round(4))

# Overall summary: average over GT-present labels
summary_df = (
    metrics_df.groupby("Model")
    [["Dice", "IoU", "Precision", "Recall",
      "HD95_mm", "ASSD_mm"]]
    .mean()
    .reset_index()
)

print("\n========== CASE 001: OVERALL SUMMARY ==========")
display(summary_df.round(4))

summary_df.to_csv(
    os.path.join(METRIC_DIR, "Case001_Overall_Metrics.csv"),
    index=False
)

print("\nSaved:", csv_path)

In [ ]:
timing_variables = [
    "coarse_inference_time",
    "refinement_time",
    "baseline_inference_time",
    "adaptive_inference_time",
    "roi_times"
]

for name in timing_variables:
    print(
        name, ":",
        globals().get(name, "Not recorded")
    )

In [ ]:
import torch

print("========== AVAILABLE PYTORCH MODELS ==========")

for name, obj in list(globals().items()):
    if isinstance(obj, torch.nn.Module):
        print(
            "Model:", name,
            "| Device:", next(obj.parameters()).device,
            "| Parameters:", sum(p.numel() for p in obj.parameters())
        )

print("\n========== AVAILABLE INPUT VARIABLES ==========")

for name in [
    "ct", "coarse_input", "coarse_tensor",
    "roi_patches", "refined_roi_probabilities"
]:
    if name in globals():
        obj = globals()[name]
        if hasattr(obj, "shape"):
            print(name, "Shape:", obj.shape)
        else:
            print(name, "Available")

print("\n========== GPU INFORMATION ==========")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

print("========== AVAILABLE PYTORCH MODELS ==========")

for name, obj in list(globals().items()):

    if isinstance(obj, torch.nn.Module):

        params = list(obj.parameters())

        if params:
            device = params[0].device
            total_params = sum(p.numel() for p in params)
        else:
            device = "No parameters"
            total_params = 0

        print("Model:", name)
        print("Device:", device)
        print("Parameters:", total_params)
        print("-----------------------------")


print("\n========== AVAILABLE INPUT VARIABLES ==========")

for name in [
    "ct",
    "coarse_input",
    "coarse_tensor",
    "roi_patches",
    "refined_roi_probabilities",
    "coarse_logits"
]:

    if name in globals():

        obj = globals()[name]

        if hasattr(obj, "shape"):
            print(name, "Shape:", obj.shape)
        else:
            print(name, "Available")


print("\n========== GPU INFORMATION ==========")

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import torch

print("========== MODEL ARCHITECTURE ==========")

print(baseline_model)

print("\n========== MODEL CONFIGURATION ==========")

print("Training model device:",
      next(baseline_model.parameters()).device)

print("Model training mode:", baseline_model.training)

print("\n========== COARSE INPUT DETAILS ==========")

print("CT shape:", ct.shape)
print("Coarse logits shape:", coarse_logits.shape)

print("\n========== AVAILABLE INFERENCE FUNCTIONS ==========")

import inspect

for name, obj in list(globals().items()):
    if callable(obj) and any(
        word in name.lower()
        for word in ["infer", "predict", "sliding", "coarse", "refine"]
    ):
        try:
            print(name, inspect.signature(obj))
        except Exception:
            print(name, "Available")

In [ ]:
import os
import numpy as np
import plotly.graph_objects as go

OUT = "/kaggle/working/PENGWIN/Paper_Plots/04_3D_Reconstruction"
os.makedirs(OUT, exist_ok=True)

colors = {
    "Sacrum": "royalblue",
    "Left Hipbone": "hotpink",
    "Right Hipbone": "tomato"
}

def add_mesh(fig, mesh, name, color, showlegend=False):
    v = mesh.vertices
    f = mesh.faces

    fig.add_trace(go.Mesh3d(
        x=v[:, 0], y=v[:, 1], z=v[:, 2],
        i=f[:, 0], j=f[:, 1], k=f[:, 2],
        color=color,
        opacity=1,
        name=name,
        showlegend=showlegend,
        hovertemplate=name + "<extra></extra>"
    ))

# FIGURE 1: Complete pelvis
fig = go.Figure()

for region, meshes in bone_groups.items():
    for idx, (filename, mesh) in enumerate(meshes):
        add_mesh(
            fig, mesh, filename, colors[region],
            showlegend=(idx == 0)
        )

fig.update_layout(
    title="Complete AI-Based 3D Pelvic Reconstruction",
    scene=dict(
        xaxis_title="X (mm)",
        yaxis_title="Y (mm)",
        zaxis_title="Z (mm)",
        aspectmode="data"
    ),
    width=1100,
    height=850
)

fig.write_html(os.path.join(OUT, "Figure_01_Complete_Pelvis.html"))
fig.show()

# FIGURES 2–4: Individual bone regions
for region, meshes in bone_groups.items():

    fig = go.Figure()

    for filename, mesh in meshes:
        add_mesh(fig, mesh, filename, colors[region])

    fig.update_layout(
        title=f"3D Reconstruction - {region}",
        scene=dict(aspectmode="data"),
        width=1000,
        height=800
    )

    filename = region.replace(" ", "_")
    fig.write_html(
        os.path.join(OUT, f"Figure_{filename}.html")
    )
    fig.show()

# FIGURE 5: All fragments with individual hover IDs
fig = go.Figure()

for region, meshes in bone_groups.items():
    for idx, (filename, mesh) in enumerate(meshes, start=1):
        fragment_id = f"{region}_F{idx:02d}"
        add_mesh(fig, mesh, fragment_id, colors[region])

fig.update_layout(
    title="Individual 3D Bone Fragment Identification – 41 Fragments",
    scene=dict(aspectmode="data"),
    width=1200,
    height=900
)

fig.write_html(
    os.path.join(OUT, "Figure_05_All_41_Fragment_IDs.html")
)
fig.show()

print("\nALL 3D RECONSTRUCTION OUTPUTS SAVED")
print("Folder:", OUT)